In [ ]:
PROFILE = "free"
HOURS = {"check": 2.0, "free": 11.3, "full": 11.3}[PROFILE]
EXTRA = ""

In [ ]:
!pip install -q "ultralytics==8.4.179"
import torch, ultralytics, sys
print(sys.version, "torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count())
print("ultralytics", ultralytics.__version__)

In [ ]:
!mkdir -p /tmp/dentx_src/dentx/experiments

In [ ]:
%%writefile /tmp/dentx_src/dentx/__init__.py
__version__ = '1.0.0'

In [ ]:
%%writefile /tmp/dentx_src/dentx/analysis.py
from __future__ import annotations
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from .config import Paths
from .corrupt import parse
from .infer import dump_path, eval_sets_for, set_ids
from .metrics import Bootstrap, SetEval, evaluate
from .train import is_done
from .utils import LOG, jload, stable_seed
SUBSETS = {'pk': ['all', 'retractor', 'no_retractor']}

class Store:

    def __init__(self, cfg: dict, paths: Paths):
        self.cfg, self.paths = (cfg, paths)
        self.man = pd.read_csv(paths.manifest, dtype={'group': str, 'condition': str})
        self.man['condition'] = self.man['condition'].fillna('')
        self.plan = jload(paths.plan)
        self.avail = sorted(self.man.dataset.unique())
        self.group = dict(zip(self.man.image_id, self.man.group))
        self.cond = dict(zip(self.man.image_id, self.man.condition))
        self._gt, self._dumps, self._se = ({}, {}, {})

    def runs(self, kind=None):
        return [r for r, p in self.plan.items() if is_done(self.paths, r) and (kind is None or p['kind'] == kind)]

    def gt(self, ds: str) -> dict:
        if ds not in self._gt:
            self._gt[ds] = jload(self.paths.gt / f'{ds}.json.gz')
        return self._gt[ds]

    def dump(self, rid: str, name: str):
        key = (rid, name)
        if key not in self._dumps:
            p = dump_path(self.paths, rid, name)
            self._dumps[key] = jload(p) if p.exists() else None
        return self._dumps[key]

    def feats(self, rid: str, name: str):
        p = dump_path(self.paths, rid, name, 'feat.npy')
        return np.load(p).astype(np.float32) if p.exists() else None

    def cam(self, rid: str, name: str):
        p = dump_path(self.paths, rid, name, 'cam.json.gz')
        return jload(p) if p.exists() else None

    @staticmethod
    def ds_of(name: str) -> str:
        base, _, _ = parse(name)
        d = base.rsplit('_', 1)[0]
        return d[:-3] if d.endswith('img') else d

    def gt_boxes(self, ds: str, ids, variant: str='lesion'):
        g = self.gt(ds)
        key = 'tooth' if variant == 'tooth' else 'boxes'
        return [g[i][key] or [] for i in ids]

    def gt_subs(self, ds: str, ids):
        g = self.gt(ds)
        return [g[i]['sub'] for i in ids]

    def filter_ids(self, ids, subset: str):
        if subset in (None, 'all'):
            return list(ids)
        return [i for i in ids if self.cond.get(i) == subset]

    def set_eval(self, rid: str, names, variant='lesion', subset='all', use_flip=False, ids_keep=None):
        names = [names] if isinstance(names, str) else list(names)
        key = (rid, tuple(names), variant, subset, use_flip, None if ids_keep is None else tuple(ids_keep))
        if key in self._se:
            return self._se[key]
        ids, preds, gts = ([], [], [])
        for nm in names:
            d = self.dump(rid, nm)
            if d is None:
                self._se[key] = None
                return None
            ds = self.ds_of(nm)
            keep = set(self.filter_ids(d['ids'], subset))
            if ids_keep is not None:
                keep &= set(ids_keep)
            P = d['pf'] if use_flip else d['p']
            sel = [(i, p) for i, p in zip(d['ids'], P) if i in keep]
            ids += [i for i, _ in sel]
            preds += [p for _, p in sel]
            gts += self.gt_boxes(ds, [i for i, _ in sel], variant)
        e = cfg_eval = self.cfg['eval']
        se = SetEval(ids, preds, gts, [self.group.get(i, i) for i in ids], iou=e['iou'], iomin=e['iomin'])
        self._se[key] = se
        return se

    def val_names(self, rid: str):
        p = self.plan[rid]
        img = 'img' if p.get('split_col') == 'split_img' else ''
        return [f'{ds}{img}_val' for ds in p['datasets']]

    def val_subset(self, rid: str) -> str:
        f = self.plan[rid].get('filter') or {}
        return f.get('condition', 'all')

    def thr_val(self, rid: str, level: str='strict') -> float:
        se = self.set_eval(rid, self.val_names(rid), subset=self.val_subset(rid))
        return se.best_threshold(level) if se is not None and se.n_gt else self.cfg['eval']['thr_fixed']

def boot_for(store: Store, se: SetEval, name: str, subset: str, variant: str) -> Bootstrap:
    return Bootstrap(se.groups, store.cfg['eval']['bootstrap'], stable_seed(name, subset, variant))

def compute_master(store: Store) -> tuple[pd.DataFrame, dict]:
    cfg = store.cfg
    E = cfg['eval']
    rows, reps = ([], {})
    for rid in store.runs('main'):
        spec = store.plan[rid]
        thr_v = store.thr_val(rid)
        for s in eval_sets_for(spec, cfg, store.avail):
            name = s['name']
            if name.endswith('_pool'):
                continue
            base, kind, sev = parse(name)
            ds = store.ds_of(name)
            clean = kind == 'clean'
            is_val = base.endswith('_val')
            subsets = SUBSETS.get(ds, ['all']) if clean and (not is_val) else ['all']
            variants = ['lesion', 'tooth'] if ds == 'dai' and clean and (not is_val) else ['lesion']
            for subset in subsets:
                for variant in variants:
                    se = store.set_eval(rid, name, variant, subset)
                    if se is None or se.n == 0:
                        continue
                    boot = boot_for(store, se, name, subset, variant) if clean and (not is_val) else None
                    for mode, thr in (('fixed', E['thr_fixed']), ('val', thr_v)):
                        m, r = evaluate(se, thr, boot, want_ap=mode == 'fixed', ap5095=clean, ece_bins=E['ece_bins'], auroc_boot=500 if boot is not None and mode == 'val' else 0)
                        row = dict(run=rid, model=spec['model'], cfg=spec['cfg'], train_datasets='+'.join(spec['datasets']), set=name, set_base=base, target=ds, corruption=kind, severity=sev, subset=subset, gt=variant, thr_mode=mode, is_val=int(is_val), in_domain=int(ds in spec['datasets']))
                        row.update(m)
                        rows.append(row)
                        for lv, arr in r.items():
                            reps[rid, name, subset, variant, mode, lv] = arr.astype(np.float32)
    df = pd.DataFrame(rows)
    LOG.info('master table: %d rows from %d runs', len(df), df.run.nunique() if len(df) else 0)
    return (df, reps)

def save_master(df: pd.DataFrame, reps: dict, paths: Paths) -> None:
    df.to_csv(paths.tables / 'all_metrics.csv', index=False)
    with open(paths.results / 'bootstrap_reps.pkl', 'wb') as f:
        pickle.dump(reps, f, protocol=4)

In [ ]:
%%writefile /tmp/dentx_src/dentx/config.py
from __future__ import annotations
import copy
import hashlib
import json
from pathlib import Path
import yaml
PKG_DIR = Path(__file__).resolve().parent
DEFAULT_CONFIG = PKG_DIR / 'config.yaml'

def on_kaggle() -> bool:
    return Path('/kaggle/working').is_dir()

def deep_merge(base: dict, over: dict) -> dict:
    out = copy.deepcopy(base)
    for k, v in (over or {}).items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = deep_merge(out[k], v)
        else:
            out[k] = copy.deepcopy(v)
    return out

def load_config(profile: str | None=None, overrides: dict | None=None, path: str | Path | None=None, work: str | None=None) -> dict:
    raw = yaml.safe_load(Path(path or DEFAULT_CONFIG).read_text(encoding='utf-8'))
    profiles = raw.pop('profiles', {}) or {}
    prof = profile or raw.get('profile', 'full')
    cfg = raw
    if prof != 'full':
        if prof not in profiles:
            raise KeyError(f"unknown profile {prof!r}; choose from full, {', '.join(profiles)}")
        cfg = deep_merge(cfg, profiles[prof])
    if overrides:
        cfg = deep_merge(cfg, overrides)
    cfg['profile'] = prof
    if work:
        cfg['paths']['work'] = work
    _resolve_paths(cfg)
    return cfg

def _resolve_paths(cfg: dict) -> None:
    p = cfg['paths']
    kaggle = on_kaggle()
    if p.get('work', 'auto') == 'auto':
        p['work'] = '/kaggle/working/dentx_run' if kaggle else str(Path.cwd() / 'runs' / cfg['profile'])
    work = Path(p['work'])
    if p.get('data', 'auto') == 'auto':
        p['data'] = '/tmp/dentx_data' if kaggle else str(work / 'data')
    if p.get('raw', 'auto') == 'auto':
        p['raw'] = '/tmp/dentx_raw' if kaggle else str(work / 'raw')
    if p.get('inputs', 'auto') == 'auto':
        p['inputs'] = ['/kaggle/input'] if kaggle else []
    p['train_tmp'] = '/tmp/dentx_train' if kaggle else str(work / 'train_tmp')
    p['pretrained'] = '/tmp/dentx_pretrained' if kaggle else str(work / 'pretrained')
    for k in ('work', 'data', 'raw', 'train_tmp', 'pretrained'):
        p[k] = str(Path(p[k]).resolve())
    p['inputs'] = [str(Path(x).resolve()) for x in p['inputs']]

def config_hash(cfg: dict, keys=('prepare', 'datasets')) -> str:
    sub = {k: cfg.get(k) for k in keys}
    return hashlib.md5(json.dumps(sub, sort_keys=True, default=str).encode()).hexdigest()[:10]

class Paths:

    def __init__(self, cfg: dict):
        p = cfg['paths']
        self.work = Path(p['work'])
        self.data = Path(p['data'])
        self.raw = Path(p['raw'])
        self.inputs = [Path(x) for x in p.get('inputs', [])]
        self.train_tmp = Path(p['train_tmp'])
        self.pretrained = Path(p['pretrained'])
        self.runs = self.work / 'runs'
        self.dumps = self.work / 'dumps'
        self.gt = self.work / 'gt'
        self.results = self.work / 'results'
        self.tables = self.results / 'tables'
        self.figures = self.results / 'figures'
        self.logs = self.work / 'logs'
        self.manifest = self.work / 'manifest.csv'
        self.plan = self.work / 'plan.json'
        self.cfg_file = self.work / 'config_resolved.yaml'

    def make(self) -> 'Paths':
        for d in (self.work, self.data, self.raw, self.train_tmp, self.pretrained, self.runs, self.dumps, self.gt, self.results, self.tables, self.figures, self.logs):
            d.mkdir(parents=True, exist_ok=True)
        return self

def save_resolved(cfg: dict, paths: Paths) -> None:
    paths.cfg_file.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')

def load_resolved(work: str | Path) -> dict:
    return yaml.safe_load((Path(work) / 'config_resolved.yaml').read_text(encoding='utf-8'))

In [ ]:
%%writefile /tmp/dentx_src/dentx/corrupt.py
from __future__ import annotations
import cv2
import numpy as np
from .utils import stable_seed
PARAMS = {'gaussian_blur': [1.0, 2.0, 3.0, 4.5, 6.0], 'brightness': [0.8, 0.65, 0.5, 0.38, 0.25], 'contrast': [0.75, 0.6, 0.45, 0.32, 0.2], 'gaussian_noise': [0.04, 0.08, 0.12, 0.18, 0.26], 'jpeg': [60, 40, 25, 15, 8]}
TYPES = list(PARAMS)

def corrupt(img: np.ndarray, kind: str, severity: int, key: str='') -> np.ndarray:
    if kind in (None, '', 'clean') or severity == 0:
        return img
    p = PARAMS[kind][severity - 1]
    scale = max(img.shape[:2]) / 640.0
    if kind == 'gaussian_blur':
        return cv2.GaussianBlur(img, (0, 0), sigmaX=p * scale)
    if kind == 'brightness':
        return np.clip(img.astype(np.float32) * p, 0, 255).astype(np.uint8)
    if kind == 'contrast':
        f = img.astype(np.float32)
        m = f.mean(axis=(0, 1), keepdims=True)
        return np.clip((f - m) * p + m, 0, 255).astype(np.uint8)
    if kind == 'gaussian_noise':
        rng = np.random.default_rng(stable_seed(key, kind, severity))
        f = img.astype(np.float32) / 255.0 + rng.normal(0, p, img.shape).astype(np.float32)
        return (np.clip(f, 0, 1) * 255).astype(np.uint8)
    if kind == 'jpeg':
        ok, buf = cv2.imencode('.jpg', img[:, :, ::-1], [int(cv2.IMWRITE_JPEG_QUALITY), int(p)])
        return cv2.imdecode(buf, cv2.IMREAD_COLOR)[:, :, ::-1].copy()
    raise ValueError(kind)

def parse(tag: str):
    if '@' not in tag:
        return (tag, 'clean', 0)
    base, c = tag.split('@', 1)
    return (base, c.rstrip('0123456789'), int(c[len(c.rstrip('0123456789')):]))

In [ ]:
%%writefile /tmp/dentx_src/dentx/download.py
from __future__ import annotations
import os
import shutil
import subprocess
import time
import tarfile
import urllib.request
import zipfile
from pathlib import Path
from .config import Paths
from .utils import LOG

def _walk(base: Path, max_depth: int=6):
    base = Path(base)
    if not base.exists():
        return
    stack = [(base, 0)]
    while stack:
        d, depth = stack.pop()
        try:
            entries = list(os.scandir(d))
        except (PermissionError, FileNotFoundError, NotADirectoryError):
            continue
        for e in entries:
            p = Path(e.path)
            yield (p, e.is_dir())
            if e.is_dir() and depth < max_depth and (not e.name.startswith('.')):
                stack.append((p, depth + 1))

def _search_bases(paths: Paths):
    return [paths.raw] + list(paths.inputs)

def find_pk(paths: Paths):
    for base in _search_bases(paths):
        for p, is_dir in _walk(base):
            if not is_dir and p.name == 'Dataset.zip' and zipfile.is_zipfile(p):
                return p
            if is_dir and p.name == 'Dataset' and (p / 'Images').is_dir() and (p / 'Annotations').is_dir():
                return p.parent
    return None

def find_ad(paths: Paths):
    for base in _search_bases(paths):
        for p, is_dir in _walk(base):
            if not is_dir and p.name == 'AlphaDent.zip' and zipfile.is_zipfile(p):
                return p
            if not is_dir and p.name == 'yolo_seg_train.yaml' and (p.parent / 'labels').is_dir():
                return p.parent
    return None

def find_dai(paths: Paths):
    for base in _search_bases(paths):
        for p, is_dir in _walk(base):
            if not is_dir and p.name == 'dentalai-DatasetNinja.tar' and tarfile.is_tarfile(p):
                return p
            if is_dir and (p / 'meta.json').exists() and any(((p / s / 'ann').is_dir() for s in ('train', 'valid', 'test'))):
                return p
    masks = None
    for base in _search_bases(paths):
        cocos, yamls = ([], [])
        for p, is_dir in _walk(base):
            if is_dir:
                continue
            if p.name == '_classes.csv' and masks is None:
                masks = p.parent.parent
            if p.name == '_annotations.coco.json':
                cocos.append(p)
            elif p.name in ('data.yaml', 'dataset.yaml'):
                try:
                    txt = p.read_text(encoding='utf-8', errors='ignore').lower()
                except OSError:
                    continue
                if 'caries' in txt and 'tooth' in txt:
                    yamls.append(p)
        if cocos:
            return Path(os.path.commonpath([str(c.parent.parent) for c in cocos]))
        if yamls:
            return yamls[0].parent
    return masks
FINDERS = {'pk': find_pk, 'ad': find_ad, 'dai': find_dai}

def _stream(url: str, dest: Path, retries: int=4) -> None:
    tmp = dest.with_name(dest.name + '.part')
    for attempt in range(1, retries + 1):
        try:
            done = tmp.stat().st_size if tmp.exists() else 0
            req = urllib.request.Request(url, headers={'Range': f'bytes={done}-'} if done else {})
            with urllib.request.urlopen(req, timeout=60) as r:
                total = r.headers.get('Content-Length')
                total = int(total) + done if total else None
                mode = 'ab' if done and r.status == 206 else 'wb'
                if mode == 'wb':
                    done = 0
                last = time.time()
                with open(tmp, mode) as f:
                    while True:
                        chunk = r.read(1 << 22)
                        if not chunk:
                            break
                        f.write(chunk)
                        done += len(chunk)
                        if time.time() - last > 30:
                            LOG.info('  %s: %.0f / %s MB', dest.name, done / 2 ** 20, f'{total / 2 ** 20:.0f}' if total else '?')
                            last = time.time()
            os.replace(tmp, dest)
            return
        except Exception as e:
            LOG.warning('download attempt %d/%d failed for %s: %s', attempt, retries, url, e)
            time.sleep(5 * attempt)
    raise RuntimeError(f'could not download {url}')

def ensure_dataset(ds: str, cfg: dict, paths: Paths):
    found = FINDERS[ds](paths)
    masks_only = ds == 'dai' and found is not None and found.is_dir() and any(found.rglob('_classes.csv'))
    if found and (not masks_only):
        LOG.info('dataset %s found at %s', ds, found)
        return found
    if masks_only:
        LOG.info('DentalAI semantic-mask copy found at %s; trying the instance-polygon download first', found)
    dcfg = cfg['datasets'][ds]
    paths.raw.mkdir(parents=True, exist_ok=True)
    if 'urls' in dcfg:
        dest = paths.raw / dcfg['archive']
        for url in dcfg['urls']:
            try:
                LOG.info('downloading %s from %s', ds, url)
                _stream(url, dest)
                if zipfile.is_zipfile(dest) or tarfile.is_tarfile(dest):
                    return dest
                LOG.warning('%s is not a valid archive; trying next mirror', dest)
                dest.unlink(missing_ok=True)
            except Exception as e:
                LOG.warning('mirror failed: %s', e)
        if masks_only:
            LOG.warning('using the DentalAI semantic-mask copy: tooth-level ground truth will be approximate')
            return found
        if not dcfg.get('kaggle_slug'):
            return None
    slug = dcfg.get('kaggle_slug')
    if slug and shutil.which('kaggle'):
        out = paths.raw / ds
        out.mkdir(parents=True, exist_ok=True)
        LOG.info('downloading %s with kaggle CLI (%s)', ds, slug)
        res = subprocess.run(['kaggle', 'datasets', 'download', '-d', slug, '-p', str(out), '--unzip'], capture_output=True, text=True)
        if res.returncode == 0:
            return FINDERS[ds](paths)
        LOG.warning('kaggle download failed: %s', res.stderr.strip()[:500])
    LOG.warning("dataset %s unavailable. On Kaggle, attach '%s' as an input.", ds, slug)
    return None

In [ ]:
%%writefile /tmp/dentx_src/dentx/experiments/__init__.py
pass

In [ ]:
%%writefile /tmp/dentx_src/dentx/experiments/common.py
from __future__ import annotations
import numpy as np
import pandas as pd

def pick(df: pd.DataFrame, **kw) -> pd.DataFrame:
    m = pd.Series(True, index=df.index)
    for k, v in kw.items():
        if isinstance(v, (list, tuple, set)):
            m &= df[k].isin(list(v))
        else:
            m &= df[k] == v
    return df[m]

def fmt_ci(v, lo, hi, nd=3) -> str:
    if v is None or not np.isfinite(v):
        return '–'
    if lo is None or not np.isfinite(lo):
        return f'{v:.{nd}f}'
    return f'{v:.{nd}f} ({lo:.{nd}f}–{hi:.{nd}f})'

def clean_test(M: pd.DataFrame, thr_mode: str='val') -> pd.DataFrame:
    return pick(M, corruption='clean', is_val=0, subset='all', gt='lesion', thr_mode=thr_mode)

def rep(reps: dict, rid, name, level='strict', mode='val', subset='all', variant='lesion'):
    return reps.get((rid, name, subset, variant, mode, level))

def single_source(store) -> list:
    out = []
    for r in store.runs('main'):
        p = store.plan[r]
        if len(p['datasets']) == 1 and (not p.get('filter')) and (p.get('split_col', 'split') == 'split') and (p['cfg'] == p['datasets'][0]):
            out.append(r)
    return out

In [ ]:
%%writefile /tmp/dentx_src/dentx/experiments/e0_replicate.py
from __future__ import annotations
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from .. import figs
from ..corrupt import parse
from ..stats import independent_diff
from ..utils import write_csv_and_tex
from .common import fmt_ci, pick, rep

def run(store, M, reps, paths) -> dict:
    figs.style()
    cfg = store.cfg
    thr = cfg['eval']['thr_fixed']
    out = {}
    if M.empty:
        return out
    models = sorted(M.model.unique())
    rows = []
    for c, test in (('pkimg20', 'pkimg_test'), ('pk20', 'pk_test'), ('pkimg', 'pkimg_test'), ('pk', 'pk_test')):
        sub = pick(M, cfg=c, set=test, subset='all', gt='lesion', thr_mode='fixed')
        for _, r in sub.iterrows():
            rows.append(dict(model=r.model, config=c, split='image-level' if 'img' in c else 'patient-level', epochs=store.plan[r.run]['epochs'], AP50=r.AP50, AP50_95=r.AP50_95, precision=r.strict_P, recall=r.strict_R, F1=fmt_ci(r.strict_F1, r.get('strict_F1_lo'), r.get('strict_F1_hi'))))
    if rows:
        t = pd.DataFrame(rows).sort_values(['model', 'config'])
        write_csv_and_tex(t, paths.tables / 'e0_replication_leakage', caption='Replication and split leakage (Pakistan test, threshold 0.25)')
        out['replication'] = t.to_dict('records')
    leak = []
    for m in models:
        for img_cfg, pat_cfg, ep in (('pkimg20', 'pk20', 20), ('pkimg', 'pk', None)):
            a = rep(reps, f'{m}__{img_cfg}', 'pkimg_test', mode='fixed')
            b = rep(reps, f'{m}__{pat_cfg}', 'pk_test', mode='fixed')
            if a is not None and b is not None:
                d = independent_diff(a, b)
                leak.append(dict(model=m, epochs=ep or store.plan[f'{m}__{pat_cfg}']['epochs'], F1_image_split_minus_patient_split=d['diff'], lo=d['lo'], hi=d['hi'], p=d['p']))
    if leak:
        write_csv_and_tex(pd.DataFrame(leak), paths.tables / 'e0_leakage_effect', caption='Leakage inflation of F1')
        out['leakage'] = leak
    rows = []
    for m in models:
        for tr, other in (('retractor', 'no_retractor'), ('no_retractor', 'retractor')):
            rid = f"{m}__pk{('R' if tr == 'retractor' else 'N')}"
            ins = pick(M, run=rid, set='pk_test', subset=tr, gt='lesion', thr_mode='fixed')
            crs = pick(M, run=rid, set='pk_test', subset=other, gt='lesion', thr_mode='fixed')
            if ins.empty or crs.empty:
                continue
            a = rep(reps, rid, 'pk_test', mode='fixed', subset=tr)
            b = rep(reps, rid, 'pk_test', mode='fixed', subset=other)
            d = independent_diff(a, b) if a is not None and b is not None else dict(diff=np.nan, lo=np.nan, hi=np.nan, p=np.nan)
            i, c = (ins.iloc[0], crs.iloc[0])
            rows.append(dict(model=m, train=tr, F1_in=fmt_ci(i.strict_F1, i.get('strict_F1_lo'), i.get('strict_F1_hi')), F1_cross=fmt_ci(c.strict_F1, c.get('strict_F1_lo'), c.get('strict_F1_hi')), gap=d['diff'], gap_lo=d['lo'], gap_hi=d['hi'], p=d['p']))
    if rows:
        write_csv_and_tex(pd.DataFrame(rows), paths.tables / 'e0_cross_condition', caption='Retractor cross-condition (patient-level split)')
        out['cross_condition'] = rows
    rows = []
    for m in models:
        rid = f'{m}__pk'
        if rid not in store.plan or rid not in set(M.run):
            continue
        for name in ['pk_test'] + [f'pk_test@{c}{s}' for c in cfg['corruptions']['types'] for s in cfg['corruptions']['severities']]:
            se = store.set_eval(rid, name)
            if se is None:
                continue
            _, kind, sev = parse(name)
            pr = se.prf(thr)
            sel = se.p_conf >= thr
            rows.append(dict(model=m, corruption=kind, severity=sev, P=pr['P'], R=pr['R'], F1=pr['F1'], n_pred=int(sel.sum()), mean_conf_tp=float(se.p_conf[sel & se.p_tp].mean()) if (sel & se.p_tp).any() else np.nan, mean_conf_fp=float(se.p_conf[sel & ~se.p_tp].mean()) if (sel & ~se.p_tp).any() else np.nan, mean_image_score=float(se.score.mean())))
    if rows:
        cd = pd.DataFrame(rows)
        write_csv_and_tex(cd, paths.tables / 'e0_corruptions', caption='Pakistan test under corruption (threshold 0.25)')
        base = cd[cd.corruption == 'clean']
        types = cfg['corruptions']['types']
        for m in cd.model.unique():
            fig, axes = plt.subplots(1, 4, figsize=(12, 2.9), sharex=True)
            b0 = base[base.model == m]
            for ax, met, lab in zip(axes, ['P', 'R', 'F1', 'n_pred'], ['Precision', 'Recall', 'F1', 'Detections ≥ thr']):
                for k, ty in enumerate(types):
                    s = cd[(cd.model == m) & (cd.corruption == ty)].sort_values('severity')
                    x = [0] + list(s.severity)
                    y = list(b0[met]) + list(s[met])
                    ax.plot(x, y, color=figs.SERIES[k], marker=figs.MARKERS[k], label=ty.replace('_', ' '))
                ax.set_title(lab)
                ax.set_xlabel('Severity (0 = clean)')
            axes[0].legend(fontsize=7, loc='lower left')
            fig.suptitle(f'{m} trained on Pakistan: degradation curves', y=1.03)
            figs.save(fig, paths.figures / f'e0_corruption_curves_{m}')
        out['corruption_rows'] = len(cd)
    fig, ax = plt.subplots(figsize=(3.6, 3.4))
    ax.plot([0, 1], [0, 1], ls='--', color=figs.INK2, lw=1, label='perfect')
    cal = []
    for m in models:
        rid = f'{m}__pk'
        se = store.set_eval(rid, 'pk_test') if rid in set(M.run) else None
        if se is None:
            continue
        e, bins = se.ece(thr, cfg['eval']['ece_bins'])
        cal.append(dict(model=m, ECE=e, n_pred=int((se.p_conf >= thr).sum())))
        if bins:
            ax.plot([b[3] for b in bins], [b[4] for b in bins], color=figs.model_color(m, models), marker=figs.model_marker(m, models), label=f'{m} (ECE {e:.3f})')
    ax.set_xlabel('Predicted confidence')
    ax.set_ylabel('Empirical precision (IoU ≥ 0.5)')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.legend(fontsize=7)
    figs.save(fig, paths.figures / 'e0_reliability_pk')
    if cal:
        write_csv_and_tex(pd.DataFrame(cal), paths.tables / 'e0_calibration', caption='Detection calibration, Pakistan test')
        out['calibration'] = cal
    rows = []
    for m in cfg['cam']['models']:
        rid = f'{m}__pk'
        cam = store.cam(rid, 'pk_test')
        se = store.set_eval(rid, 'pk_test') if cam else None
        if not cam or se is None:
            continue
        tp_imgs = set(np.array(se.ids)[np.unique(se.p_img[(se.p_conf >= thr) & se.p_tp])])
        v = np.array([x for i, x in cam['eibr'].items() if i in tp_imgs and np.isfinite(x)])
        if len(v):
            rows.append(dict(model=m, n_images=len(v), EIBR_mean=v.mean(), EIBR_median=np.median(v), EIBR_min=v.min(), EIBR_max=v.max()))
        _cam_figure(store, cam, tp_imgs, m, paths, cfg['cam']['n_examples'])
    if rows:
        write_csv_and_tex(pd.DataFrame(rows), paths.tables / 'e0_eigencam_eibr', caption='EigenCAM energy-in-box ratio')
        out['eigencam'] = rows
    return out

def _cam_figure(store, cam, tp_imgs, model, paths, n):
    ex = [i for i in cam['examples'] if i in tp_imgs][:n]
    if not ex:
        return
    path = dict(zip(store.man.image_id, store.man.image))
    fig, axes = plt.subplots(2, len(ex), figsize=(2.2 * len(ex), 4.2), squeeze=False)
    gt = store.gt('pk')
    for k, iid in enumerate(ex):
        im = cv2.imread(path.get(iid, ''))
        if im is None:
            plt.close(fig)
            return
        im = im[:, :, ::-1]
        H, W = im.shape[:2]
        for row in (0, 1):
            ax = axes[row, k]
            ax.imshow(im)
            ax.axis('off')
            for x1, y1, x2, y2 in gt[iid]['boxes']:
                ax.add_patch(plt.Rectangle((x1 * W, y1 * H), (x2 - x1) * W, (y2 - y1) * H, fill=False, color='#1baf7a', lw=1.5))
        c = cv2.resize(np.array(cam['examples'][iid], dtype=np.float32), (W, H))
        axes[1, k].imshow(c, cmap='inferno', alpha=0.45)
        axes[1, k].set_title(f"EIBR {cam['eibr'][iid]:.2f}", fontsize=7)
    fig.suptitle(f'{model}: ground truth (green) and EigenCAM', y=1.0)
    figs.save(fig, paths.figures / f'e0_eigencam_{model}')

In [ ]:
%%writefile /tmp/dentx_src/dentx/experiments/e1_indomain.py
from __future__ import annotations
import pandas as pd
from ..stats import holm, paired_diff
from ..utils import write_csv_and_tex
from .common import clean_test, fmt_ci, pick, rep, single_source

def run(store, M, reps, paths) -> dict:
    runs = single_source(store)
    if M.empty or not runs:
        return {}
    rows, comp = ([], [])
    for ds in store.avail:
        rs = [r for r in runs if store.plan[r]['datasets'] == [ds]]
        sub = pick(clean_test(M, 'val'), set=f'{ds}_test')
        sub = sub[sub.run.isin(rs)]
        fixed = pick(clean_test(M, 'fixed'), set=f'{ds}_test')
        if sub.empty:
            continue
        for _, r in sub.iterrows():
            f = fixed[fixed.run == r.run]
            rows.append(dict(dataset=ds, model=r.model, AP50=f.AP50.iloc[0] if len(f) else None, AP50_95=f.AP50_95.iloc[0] if len(f) else None, thr=r.thr, F1_strict=fmt_ci(r.strict_F1, r.get('strict_F1_lo'), r.get('strict_F1_hi')), F1_lenient=fmt_ci(r.lenient_F1, r.get('lenient_F1_lo'), r.get('lenient_F1_hi')), sens_image=fmt_ci(r.image_R, r.get('image_R_lo'), r.get('image_R_hi')), spec_image=fmt_ci(r.image_spec, r.get('image_spec_lo'), r.get('image_spec_hi')), AUROC_image=fmt_ci(r.img_auc, r.get('img_auc_lo'), r.get('img_auc_hi')), _f1=r.strict_F1))
        best = sub.sort_values('strict_F1', ascending=False).iloc[0]
        tmp = []
        for _, r in sub.iterrows():
            if r.run == best.run:
                continue
            a, b = (rep(reps, best.run, f'{ds}_test'), rep(reps, r.run, f'{ds}_test'))
            if a is None or b is None:
                continue
            d = paired_diff(a, b)
            tmp.append(dict(dataset=ds, best=best.model, other=r.model, F1_diff=d['diff'], lo=d['lo'], hi=d['hi'], p=d['p']))
        if tmp:
            adj = holm([t['p'] for t in tmp])
            for t, a in zip(tmp, adj):
                t['p_holm'] = a
            comp += tmp
    t = pd.DataFrame(rows).sort_values(['dataset', '_f1'], ascending=[True, False]).drop(columns='_f1')
    write_csv_and_tex(t, paths.tables / 'e1_indomain', caption='In-domain performance (thresholds chosen on validation)')
    if comp:
        write_csv_and_tex(pd.DataFrame(comp), paths.tables / 'e1_model_comparison', caption='Paired patient-clustered bootstrap: best model minus others (strict F1)')
    return dict(indomain=t.to_dict('records'), comparisons=comp)

In [ ]:
%%writefile /tmp/dentx_src/dentx/experiments/e2_lodo.py
from __future__ import annotations
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from .. import figs
from ..stats import friedman_nemenyi, independent_diff
from ..utils import write_csv_and_tex
from .common import clean_test, pick, rep, single_source
METRICS = [('strict_F1', 'F1 strict (IoU≥0.5)'), ('lenient_F1', 'F1 lenient (IoMin≥0.5)'), ('img_auc', 'Image-level AUROC')]

def _main_runs(store):
    return [r for r in store.runs('main') if not store.plan[r].get('filter') and store.plan[r].get('split_col', 'split') == 'split']

def run(store, M, reps, paths) -> dict:
    figs.style()
    out = {}
    if M.empty:
        return out
    runs = _main_runs(store)
    C = clean_test(M, 'val')
    C = C[C.run.isin(runs) & C.set.str.endswith('_test') & ~C.set.str.startswith('pkimg')]
    if C.empty:
        return out
    C = C.assign(external=(C.in_domain == 0).astype(int))
    keep = ['model', 'cfg', 'target', 'external', 'thr', 'strict_P', 'strict_R', 'strict_F1', 'strict_F1_lo', 'strict_F1_hi', 'lenient_F1', 'lenient_F1_lo', 'lenient_F1_hi', 'image_R', 'image_spec', 'img_auc', 'img_auc_lo', 'img_auc_hi']
    long = C[[k for k in keep if k in C.columns]].sort_values(['model', 'cfg', 'target'])
    tooth = pick(M, corruption='clean', is_val=0, subset='all', gt='tooth', thr_mode='val', target='dai')
    tooth = tooth[tooth.run.isin(runs)][['model', 'cfg', 'strict_F1', 'strict_F1_lo', 'strict_F1_hi']].rename(columns={'strict_F1': 'dai_tooth_F1', 'strict_F1_lo': 'dai_tooth_F1_lo', 'strict_F1_hi': 'dai_tooth_F1_hi'})
    write_csv_and_tex(long, paths.tables / 'e2_external_matrix_long', caption='All train-config x test-set cells')
    if not tooth.empty:
        write_csv_and_tex(tooth, paths.tables / 'e2_dai_tooth_level', caption='DentalAI test against tooth-level (harmonised) ground truth')
    models = [m for m in figs.MODEL_ORDER if m in set(C.model)] + sorted(set(C.model) - set(figs.MODEL_ORDER))
    cfgs = [c for c in ['pk', 'ad', 'dai', 'pk-ad', 'pk-dai', 'ad-dai'] if c in set(C.cfg)]
    tg = [d for d in ['pk', 'ad', 'dai'] if d in set(C.target)]
    for met, lab in METRICS:
        fig, axes = plt.subplots(1, len(models), figsize=(2.3 * len(models) + 0.6, 0.45 * len(cfgs) + 1.2), squeeze=False)
        for ax, m in zip(axes[0], models):
            grid = np.full((len(cfgs), len(tg)), np.nan)
            for i, c in enumerate(cfgs):
                for j, t in enumerate(tg):
                    v = C[(C.model == m) & (C.cfg == c) & (C.target == t)][met]
                    if len(v):
                        grid[i, j] = v.iloc[0]
            ax.imshow(grid, cmap=plt.matplotlib.colors.LinearSegmentedColormap.from_list('seq', figs.SEQ), vmin=0, vmax=1)
            for i in range(len(cfgs)):
                for j in range(len(tg)):
                    if np.isfinite(grid[i, j]):
                        indom = tg[j] in cfgs[i].split('-')
                        ax.text(j, i, f'{grid[i, j]:.2f}' + ('*' if indom else ''), ha='center', va='center', fontsize=7, color='white' if grid[i, j] > 0.55 else figs.INK)
            ax.set_xticks(range(len(tg)), [figs.DS_LABEL.get(t, t) for t in tg], rotation=30, fontsize=7)
            ax.set_yticks(range(len(cfgs)), cfgs, fontsize=7)
            ax.set_title(m, fontsize=9)
            ax.grid(False)
        fig.suptitle(f'{lab}: training set (rows) x test set (columns); * = in-domain', y=1.04, fontsize=9)
        figs.save(fig, paths.figures / f'e2_heatmap_{met}')
    gaps = []
    for rid in single_source(store):
        src = store.plan[rid]['datasets'][0]
        for t in tg:
            if t == src:
                continue
            for lv, key in (('strict', 'strict_F1'), ('lenient', 'lenient_F1'), ('image', 'image_F1')):
                a, b = (rep(reps, rid, f'{src}_test', lv), rep(reps, rid, f'{t}_test', lv))
                if a is None or b is None:
                    continue
                d = independent_diff(a, b)
                gaps.append(dict(model=store.plan[rid]['model'], source=src, target=t, level=lv, gap=d['diff'], lo=d['lo'], hi=d['hi'], p=d['p']))
    if gaps:
        write_csv_and_tex(pd.DataFrame(gaps), paths.tables / 'e2_generalisation_gap', caption='In-domain minus external F1')
        out['gaps'] = gaps
    ext = C[C.external == 1]
    rank_rows = []
    for met, _ in METRICS:
        res, _ = friedman_nemenyi(ext, ['cfg', 'target'], 'model', met)
        rank_rows.append(dict(metric=met, n_cells=res['n_blocks'], k=res['k'], chi2=res['chi2'], p=res['p'], CD=res['cd'], mean_ranks='; '.join((f'{k}={v:.2f}' for k, v in res['ranks'].items()))))
    write_csv_and_tex(pd.DataFrame(rank_rows), paths.tables / 'e2_friedman', caption='Friedman test over external cells (lower rank = better)')
    out['friedman'] = rank_rows
    multi = []
    for m in models:
        for t in tg:
            pair = '-'.join((d for d in ['pk', 'ad', 'dai'] if d != t and d in tg))
            pr = C[(C.model == m) & (C.cfg == pair) & (C.target == t)]
            if pr.empty:
                continue
            for s in pair.split('-'):
                sr = C[(C.model == m) & (C.cfg == s) & (C.target == t)]
                if sr.empty:
                    continue
                for met, _ in METRICS:
                    multi.append(dict(model=m, target=t, pair=pair, single=s, metric=met, pair_value=pr[met].iloc[0], single_value=sr[met].iloc[0], gain=pr[met].iloc[0] - sr[met].iloc[0]))
    if multi:
        write_csv_and_tex(pd.DataFrame(multi), paths.tables / 'e2_second_source_gain', caption='Two-source vs one-source training on the unseen dataset')
        out['second_source'] = len(multi)
    sub_rows = []
    for rid in single_source(store):
        thr = store.thr_val(rid)
        for t in tg:
            se = store.set_eval(rid, f'{t}_test')
            if se is None:
                continue
            subs = [s for lst in store.gt_subs(t, se.ids) for s in lst]
            if len(subs) != len(se.g_max):
                continue
            hit = se.g_max >= thr
            acc = defaultdict(lambda: [0, 0])
            for s, h in zip(subs, hit):
                acc[s][0] += int(h)
                acc[s][1] += 1
            for s, (h, n) in acc.items():
                sub_rows.append(dict(model=store.plan[rid]['model'], source=store.plan[rid]['datasets'][0], target=t, subclass=s, n_lesions=n, recall_lenient=h / n))
    if sub_rows:
        write_csv_and_tex(pd.DataFrame(sub_rows), paths.tables / 'e2_recall_by_subclass', caption='Recall by lesion subclass')
        out['subclass_rows'] = len(sub_rows)
    return out

In [ ]:
%%writefile /tmp/dentx_src/dentx/experiments/e3_protocol.py
from __future__ import annotations
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from .. import figs
from ..utils import write_csv_and_tex
from .common import clean_test, pick, single_source

def run(store, M, reps, paths) -> dict:
    figs.style()
    if M.empty:
        return {}
    C = clean_test(M, 'val')
    T = pick(M, corruption='clean', is_val=0, subset='all', gt='tooth', thr_mode='val')
    rows = []
    for rid in single_source(store):
        src = store.plan[rid]['datasets'][0]
        ind = C[(C.run == rid) & (C.set == f'{src}_test')]
        if ind.empty:
            continue
        i = ind.iloc[0]
        for t in store.avail:
            if t == src:
                continue
            ex = C[(C.run == rid) & (C.set == f'{t}_test')]
            if ex.empty:
                continue
            e = ex.iloc[0]
            r = dict(model=i.model, source=src, target=t, in_strict=i.strict_F1, in_lenient=i.lenient_F1, in_image=i.image_F1, ext_strict=e.strict_F1, ext_lenient=e.lenient_F1, ext_image=e.image_F1)
            r['drop_strict'] = r['in_strict'] - r['ext_strict']
            r['drop_lenient'] = r['in_lenient'] - r['ext_lenient']
            r['drop_image'] = r['in_image'] - r['ext_image']
            tt = T[(T.run == rid) & (T.set == f'{t}_test')]
            if t == 'dai' and len(tt):
                r['ext_tooth_strict'] = tt.iloc[0].strict_F1
                r['drop_tooth_strict'] = r['in_strict'] - r['ext_tooth_strict']
            d = r['drop_strict']
            r['convention_part'] = d - r['drop_lenient']
            ok = d > 0.02 and 0 <= r['drop_lenient'] <= d
            r['share_convention'] = r['convention_part'] / d if ok else np.nan
            rows.append(r)
    if not rows:
        return {}
    df = pd.DataFrame(rows)
    write_csv_and_tex(df, paths.tables / 'e3_protocol_decomposition', caption='External F1 drop by evaluation level')
    agg = df.groupby(['source', 'target']).agg(drop_strict=('drop_strict', 'mean'), drop_lenient=('drop_lenient', 'mean'), drop_image=('drop_image', 'mean'), convention_part=('convention_part', 'mean'), share_convention=('share_convention', 'median'), n_models=('model', 'nunique')).reset_index()
    write_csv_and_tex(agg, paths.tables / 'e3_protocol_summary', caption='Mean drop across models per transfer direction')
    pairs = list(agg.itertuples())
    levels = [('drop_strict', 'strict box'), ('drop_lenient', 'lenient box'), ('drop_image', 'image decision')]
    fig, ax = plt.subplots(figsize=(1.5 + 1.25 * len(pairs), 3.2))
    w = 0.26
    for k, (col, lab) in enumerate(levels):
        x = np.arange(len(pairs)) + (k - 1) * w
        ax.bar(x, [getattr(p, col) for p in pairs], width=w - 0.03, color=figs.SERIES[k], label=lab)
        for j, p in enumerate(pairs):
            pts = df[(df.source == p.source) & (df.target == p.target)][col]
            ax.scatter(np.full(len(pts), x[j]), pts, s=9, color=figs.INK, zorder=3, linewidths=0)
    ax.axhline(0, color=figs.INK2, lw=0.8)
    ax.set_xticks(range(len(pairs)), [f'{figs.DS_LABEL.get(p.source, p.source)}→\n{figs.DS_LABEL.get(p.target, p.target)}' for p in pairs], fontsize=7)
    ax.set_ylabel('F1 drop (in-domain − external)')
    ax.set_title('External F1 drop at three evaluation levels (dots = individual models)', fontsize=9)
    ax.legend(fontsize=7)
    figs.save(fig, paths.figures / 'e3_protocol_decomposition')
    return dict(decomposition=agg.to_dict('records'))

In [ ]:
%%writefile /tmp/dentx_src/dentx/experiments/e4_labelfree.py
from __future__ import annotations
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import linalg
from .. import figs
from ..corrupt import parse
from ..metrics import SetEval, weighted_auroc
from ..stats import spearman_ci
from ..utils import write_csv_and_tex
from .common import single_source
ESTIMATORS = ['mean_max_conf', 'mean_det_conf', 'pos_rate_ratio', 'dets_ratio', 'neg_entropy', 'atc_precision', 'flip_agreement', 'neg_frechet', 'neg_mmd']
TRUTHS = ['strict_F1', 'lenient_F1', 'img_auc']

def _pca(X, k):
    mu = X.mean(0)
    _, _, vt = np.linalg.svd(X - mu, full_matrices=False)
    return (mu, vt[:k])

def frechet(A, B):
    if len(A) < 3 or len(B) < 3:
        return np.nan
    m1, m2 = (A.mean(0), B.mean(0))
    s1, s2 = (np.cov(A, rowvar=False), np.cov(B, rowvar=False))
    cs, _ = linalg.sqrtm(s1 @ s2, disp=False)
    cs = cs.real
    return float(((m1 - m2) ** 2).sum() + np.trace(s1 + s2 - 2 * cs))

def mmd2(A, B, n=400, seed=0):
    rng = np.random.default_rng(seed)
    A = A[rng.permutation(len(A))[:n]]
    B = B[rng.permutation(len(B))[:n]]
    Z = np.concatenate([A, B])
    d = ((Z[:, None] - Z[None]) ** 2).sum(-1)
    med = np.median(d[d > 0]) if (d > 0).any() else 1.0
    K = np.exp(-d / med)
    a, b = (len(A), len(B))
    return float(K[:a, :a].mean() + K[a:, a:].mean() - 2 * K[:a, a:].mean())

def _entropy(c):
    c = np.clip(c, 1e-06, 1 - 1e-06)
    return -(c * np.log(c) + (1 - c) * np.log(1 - c))

def _flip_agreement(d, thr):
    if 'pf' not in d:
        return np.nan
    a = [[b for b in p if b[4] >= thr] for p in d['p']]
    g = [[b[:4] for b in p if b[4] >= thr] for p in d['pf']]
    se = SetEval(d['ids'], a, g)
    return se.prf(thr)['F1']

def build_rows(store) -> pd.DataFrame:
    cfg = store.cfg
    rows = []
    for rid in single_source(store):
        spec = store.plan[rid]
        src = spec['datasets'][0]
        thr = store.thr_val(rid)
        val_name = f'{src}_val'
        vse = store.set_eval(rid, val_name)
        vd = store.dump(rid, val_name)
        if vse is None or vd is None:
            continue
        vconf = vse.p_conf[vse.p_conf >= thr]
        vP = vse.prf(thr)['P']
        t_atc = np.quantile(vconf, 1 - vP) if len(vconf) and np.isfinite(vP) else thr
        v_pos = (vse.score >= thr).mean()
        v_dets = (vse.p_conf >= thr).sum() / max(1, vse.n)
        vf = store.feats(rid, val_name)
        pca = _pca(vf, min(32, len(vf) - 1, vf.shape[1])) if vf is not None and len(vf) > 3 else None
        vproj = (vf - pca[0]) @ pca[1].T if pca else None
        names = [f'{t}_test' for t in store.avail]
        names += [f'{t}_test@{c}{s}' for t in store.avail for c in cfg['corruptions']['types'] for s in cfg['corruptions']['severities']]
        for name in names:
            se = store.set_eval(rid, name)
            d = store.dump(rid, name)
            if se is None or se.n == 0:
                continue
            base, kind, sev = parse(name)
            tgt = store.ds_of(name)
            shift = ('none' if kind == 'clean' else 'synthetic') if tgt == src else 'real' if kind == 'clean' else 'real+synthetic'
            c = se.p_conf[se.p_conf >= thr]
            r = dict(run=rid, model=spec['model'], source=src, target=tgt, corruption=kind, severity=sev, shift=shift, strict_F1=se.prf(thr)['F1'], strict_P=se.prf(thr)['P'], lenient_F1=se.prf(thr, 'lenient')['F1'], img_auc=se.auroc(), img_sens=se.prf(thr, 'image')['R'], mean_max_conf=float(se.score.mean()), mean_det_conf=float(c.mean()) if len(c) else 0.0, pos_rate_ratio=float((se.score >= thr).mean() / v_pos) if v_pos else np.nan, dets_ratio=float(len(c) / max(1, se.n) / v_dets) if v_dets else np.nan, neg_entropy=-float(_entropy(se.p_conf[se.p_conf >= 0.05]).mean()) if (se.p_conf >= 0.05).any() else 0.0, atc_precision=float((c >= t_atc).mean()) if len(c) else np.nan, flip_agreement=_flip_agreement(d, thr))
            f = store.feats(rid, name)
            if pca is not None and f is not None and (len(f) > 3):
                proj = (f - pca[0]) @ pca[1].T
                r['neg_frechet'] = -frechet(vproj, proj)
                r['neg_mmd'] = -mmd2(vproj, proj)
            else:
                r['neg_frechet'] = r['neg_mmd'] = np.nan
            rows.append(r)
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    ref = df[df['shift'] == 'none'].set_index('run')
    for t in TRUTHS:
        df[f'ref_{t}'] = df.run.map(ref[t]) if len(ref) else np.nan
        df[f'drop_{t}'] = df[f'ref_{t}'] - df[t]
    return df

def run(store, M, reps, paths) -> dict:
    figs.style()
    df = build_rows(store)
    if df.empty:
        return {}
    write_csv_and_tex(df, paths.tables / 'e4_labelfree_rows', caption='Label-free estimators and true metrics per shifted set')
    est = [e for e in ESTIMATORS if df[e].notna().sum() >= 4]
    syn = df[df['shift'].isin(['none', 'synthetic'])]
    real = df[df['shift'].isin(['none', 'real'])]
    rows = []
    for e in est:
        for t in TRUTHS:
            for nm, sub in (('synthetic', syn), ('real', real), ('all', df)):
                s = spearman_ci(sub[e], sub[t])
                rows.append(dict(estimator=e, truth=t, shift=nm, **s))
    sp = pd.DataFrame(rows)
    write_csv_and_tex(sp, paths.tables / 'e4_spearman', caption='Spearman correlation of label-free estimators with true metrics')
    tr_rows, fail_rows = ([], [])
    failure = store.cfg['eval']['failure_drop']
    for t in TRUTHS:
        targets = df[df['shift'].isin(['real', 'real+synthetic'])]
        if targets.empty:
            continue
        base_err = (targets[f'ref_{t}'] - targets[t]).abs()
        tr_rows.append(dict(truth=t, estimator='no_change_baseline', MAE=base_err.mean(), n=len(targets)))
        for e in est:
            errs = []
            for rid, g in df.groupby('run'):
                s = g[g['shift'].isin(['none', 'synthetic'])].dropna(subset=[e, t])
                q = g[g['shift'].isin(['real', 'real+synthetic'])].dropna(subset=[e, t])
                if len(s) < 3 or q.empty or s[e].std() == 0:
                    continue
                b, a = np.polyfit(s[e], s[t], 1)
                errs += list((a + b * q[e] - q[t]).abs())
            if errs:
                tr_rows.append(dict(truth=t, estimator=e, MAE=float(np.mean(errs)), n=len(errs)))
            rr = sp[(sp.estimator == e) & (sp.truth == t) & (sp['shift'] == 'synthetic')].rho
            rho = rr.iloc[0] if len(rr) else np.nan
            sgn = np.sign(rho) if np.isfinite(rho) and rho != 0 else 1.0
            lab = (df[f'drop_{t}'] > failure)[df['shift'] != 'none']
            sc = (-sgn * df[e])[df['shift'] != 'none']
            ok = sc.notna()
            fail_rows.append(dict(truth=t, estimator=e, failure_AUROC=weighted_auroc(sc[ok].values, lab[ok].values), n_fail=int(lab[ok].sum()), n=int(ok.sum())))
    if tr_rows:
        write_csv_and_tex(pd.DataFrame(tr_rows), paths.tables / 'e4_transfer_mae', caption='Synthetic-fitted estimator applied to real shifts')
    if fail_rows:
        write_csv_and_tex(pd.DataFrame(fail_rows), paths.tables / 'e4_failure_detection', caption='Detecting deployment failure without labels')
    mod = []
    for rid, g in df.groupby('run'):
        ref = g[g['shift'] == 'none']
        if ref.empty:
            continue
        for t in ('strict_F1', 'lenient_F1'):
            r0 = ref[t].iloc[0]
            if not r0:
                continue
            mod.append(dict(run=rid, model=g.model.iloc[0], source=g.source.iloc[0], truth=t, synthetic_retention=g[g['shift'] == 'synthetic'][t].mean() / r0, real_retention=g[g['shift'] == 'real'][t].mean() / r0))
    md = pd.DataFrame(mod)
    out = dict(n_rows=len(df))
    if not md.empty:
        write_csv_and_tex(md, paths.tables / 'e4_robustness_vs_generalisation', caption='Corruption robustness vs external retention')
        res = []
        for t, g in md.groupby('truth'):
            res.append(dict(truth=t, **spearman_ci(g.synthetic_retention, g.real_retention)))
        write_csv_and_tex(pd.DataFrame(res), paths.tables / 'e4_robustness_spearman', caption='Does synthetic robustness predict real generalisation?')
        out['robustness_spearman'] = res
        models = sorted(md.model.unique())
        g = md[md.truth == 'strict_F1']
        sources = [d for d in ['pk', 'ad', 'dai'] if d in set(g.source)]
        rho = next((r for r in res if r['truth'] == 'strict_F1'), {}).get('rho', np.nan)
        fig, ax = plt.subplots(figsize=(4.4, 3.6))
        for k, src in enumerate(sources):
            for m in models:
                s = g[(g.source == src) & (g.model == m)]
                if len(s):
                    ax.scatter(s.synthetic_retention, s.real_retention, color=figs.SERIES[k], marker=figs.model_marker(m, models), s=46, edgecolors='white', linewidths=0.6)
        for k, src in enumerate(sources):
            ax.scatter([], [], color=figs.SERIES[k], marker='o', s=40, label=f'trained on {figs.DS_LABEL.get(src, src)}')
        for m in models:
            ax.scatter([], [], color=figs.INK2, marker=figs.model_marker(m, models), s=36, label=m)
        ax.set_xlabel('Share of F1 kept under corruption (own test set)')
        ax.set_ylabel('Share of F1 kept on external datasets')
        ax.set_title(f'Corruption robustness does not predict external transfer (ρ = {rho:.2f}, n = {len(g)})', fontsize=8)
        ax.legend(fontsize=6.5, loc='upper right')
        figs.save(fig, paths.figures / 'e4_robustness_vs_generalisation')
    cand = sp[(sp.truth == 'strict_F1') & (sp['shift'] == 'real')].dropna(subset=['rho'])
    if len(cand):
        e = cand.iloc[cand.rho.abs().argmax()].estimator
        fig, ax = plt.subplots(figsize=(3.8, 3.3))
        for k, sh in enumerate(['none', 'synthetic', 'real', 'real+synthetic']):
            s = df[df['shift'] == sh]
            ax.scatter(s[e], s.strict_F1, s=14 if sh != 'none' else 40, color=figs.SERIES[k], marker=figs.MARKERS[k], label=sh)
        ax.set_xlabel(e.replace('_', ' '))
        ax.set_ylabel('True F1 (strict)')
        ax.legend(fontsize=7)
        figs.save(fig, paths.figures / 'e4_best_estimator_scatter')
        out['best_real_estimator'] = e
    out['spearman_real_strict'] = cand.sort_values('rho', key=np.abs, ascending=False).head(5).to_dict('records') if len(cand) else []
    return out

In [ ]:
%%writefile /tmp/dentx_src/dentx/experiments/e5_fewshot.py
from __future__ import annotations
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from .. import figs
from ..train import fewshot_ids
from ..utils import write_csv_and_tex

def conformal_threshold(pos_scores, alpha):
    s = np.sort(np.asarray(pos_scores, float))
    n = len(s)
    k = int(np.floor(alpha * (n + 1)))
    return float(s[k - 1]) if k >= 1 else 0.0

def _img_metrics(se, tau):
    yes = se.score >= tau
    pos = se.pos
    sens = (yes & pos).sum() / max(1, pos.sum())
    spec = (~yes & ~pos).sum() / max(1, (~pos).sum())
    return (float(sens), float(spec))

def run(store, M, reps, paths) -> dict:
    figs.style()
    fs = store.cfg['fewshot']
    E = store.cfg['eval']
    alphas = fs.get('alphas') or [fs['alpha']]
    rows = []
    for m in fs['models']:
        src_run = f"{m}__{fs['source']}"
        if src_run not in store.runs('main'):
            continue
        thr0 = store.thr_val(src_run)
        vse = store.set_eval(src_run, f"{fs['source']}_val")
        for tgt in fs['targets']:
            tse = store.set_eval(src_run, f'{tgt}_test')
            pse_all = store.dump(src_run, f'{tgt}_pool')
            if tse is None:
                continue
            p0 = tse.prf(thr0)
            s0, sp0 = _img_metrics(tse, thr0)
            rows.append(dict(model=m, target=tgt, k=0, seed=-1, method='zero-shot', thr=thr0, F1_strict=p0['F1'], F1_lenient=tse.prf(thr0, 'lenient')['F1'], sens=s0, spec=sp0, AP50=np.nan))
            for a in alphas:
                tau_src = conformal_threshold(vse.score[vse.pos], a) if vse is not None else 0.0
                ss, sps = _img_metrics(tse, tau_src)
                rows.append(dict(model=m, target=tgt, k=0, seed=-1, method='conformal-source', alpha=a, thr=tau_src, F1_strict=np.nan, F1_lenient=np.nan, sens=ss, spec=sps, AP50=np.nan))
            for k in fs['ks']:
                for s in fs['seeds']:
                    ids = fewshot_ids(store.man, tgt, k, s)
                    if pse_all is not None:
                        kse = store.set_eval(src_run, f'{tgt}_pool', ids_keep=ids)
                        if kse is not None and kse.n:
                            thr_k = kse.best_threshold('lenient') if kse.n_gt else thr0
                            pk = tse.prf(thr_k)
                            sk, spk = _img_metrics(tse, thr_k)
                            rows.append(dict(model=m, target=tgt, k=k, seed=s, method='recalibrated', thr=thr_k, F1_strict=pk['F1'], F1_lenient=tse.prf(thr_k, 'lenient')['F1'], sens=sk, spec=spk, AP50=np.nan))
                            for a in alphas:
                                tau_k = conformal_threshold(kse.score[kse.pos], a)
                                sc, spc = _img_metrics(tse, tau_k)
                                rows.append(dict(model=m, target=tgt, k=k, seed=s, method='conformal-target', alpha=a, thr=tau_k, F1_strict=np.nan, F1_lenient=np.nan, sens=sc, spec=spc, AP50=np.nan, n_cal_pos=int(kse.pos.sum()), cal_pos_no_detection=float((kse.score[kse.pos] <= 0).mean())))
                    fr = f"fs__{m}__{fs['source']}2{tgt}__k{k}__s{s}"
                    if fr in store.runs('fewshot'):
                        fse = store.set_eval(fr, f'{tgt}_test')
                        if fse is not None:
                            t = E['thr_fixed']
                            pf = fse.prf(t)
                            sf, spf = _img_metrics(fse, t)
                            rows.append(dict(model=m, target=tgt, k=k, seed=s, method='fine-tuned', thr=t, F1_strict=pf['F1'], F1_lenient=fse.prf(t, 'lenient')['F1'], sens=sf, spec=spf, AP50=fse.ap((0.5,))[0]))
    if not rows:
        return {}
    df = pd.DataFrame(rows)
    if 'alpha' not in df:
        df['alpha'] = np.nan
    df['alpha'] = df['alpha'].fillna(-1.0)
    write_csv_and_tex(df, paths.tables / 'e5_fewshot_rows', caption='Few-shot onboarding, every draw')
    keys = ['model', 'target', 'method', 'alpha', 'k']
    agg = df.groupby(keys).agg(F1_strict=('F1_strict', 'mean'), F1_lenient=('F1_lenient', 'mean'), F1_lenient_sd=('F1_lenient', 'std'), sens=('sens', 'mean'), spec=('spec', 'mean'), AP50=('AP50', 'mean'), n=('sens', 'size')).reset_index()
    met = df.assign(met=(df.sens >= 1 - df.alpha) & (df.alpha > 0))
    cov = met[met.alpha > 0].groupby(keys)['met'].mean().rename('coverage').reset_index()
    agg = agg.merge(cov, on=keys, how='left')
    write_csv_and_tex(agg, paths.tables / 'e5_fewshot_summary', caption='Few-shot onboarding (mean over draws)')
    for m in df.model.unique():
        tg = list(df[df.model == m].target.unique())
        fig, axes = plt.subplots(2, len(tg), figsize=(3.4 * len(tg), 5.6), squeeze=False)
        for j, t in enumerate(tg):
            a = agg[(agg.model == m) & (agg.target == t)]
            ax = axes[0, j]
            for k, meth in enumerate(['zero-shot', 'recalibrated', 'fine-tuned']):
                s = a[a.method == meth].sort_values('k')
                if meth == 'zero-shot' and len(s):
                    ax.axhline(s.F1_lenient.iloc[0], color=figs.SERIES[k], ls='--', lw=1.5, label=meth)
                elif len(s):
                    ax.errorbar(s.k, s.F1_lenient, yerr=s.F1_lenient_sd.fillna(0), color=figs.SERIES[k], marker=figs.MARKERS[k], capsize=2, label=meth)
            ax.set_title(f'{figs.DS_LABEL.get(t, t)}: F1 (lenient)')
            ax.set_xlabel('k labelled target photos')
            ax.legend(fontsize=7)
            ax = axes[1, j]
            alpha = max(alphas)
            src = a[(a.method == 'conformal-source') & np.isclose(a.alpha, alpha)]
            tgt_ = a[(a.method == 'conformal-target') & np.isclose(a.alpha, alpha)].sort_values('k')
            if len(src):
                ax.axhline(src.sens.iloc[0], color=figs.SERIES[3], ls='--', lw=1.5, label='sensitivity, calibrated on source')
            if len(tgt_):
                ax.plot(tgt_.k, tgt_.sens, color=figs.SERIES[4], marker=figs.MARKERS[4], label='sensitivity, calibrated on k target photos')
                ax.plot(tgt_.k, tgt_.spec, color=figs.SERIES[6], marker=figs.MARKERS[6], ls='--', label='specificity, calibrated on k target photos')
            ax.axhline(1 - alpha, color=figs.INK2, lw=1, ls=':', label=f'guaranteed sensitivity {1 - alpha:.2f}')
            ax.set_ylim(0, 1.05)
            ax.set_title(f'{figs.DS_LABEL.get(t, t)}: conformal screening threshold')
            ax.set_xlabel('k labelled target photos')
            ax.legend(fontsize=7)
        figs.save(fig, paths.figures / f'e5_fewshot_{m}')
    return dict(summary=agg.to_dict('records'))

In [ ]:
%%writefile /tmp/dentx_src/dentx/figs.py
from __future__ import annotations
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']
MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', '*']
INK, INK2, GRID = ('#0b0b0b', '#52514e', '#e4e3df')
SEQ = ['#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#184f95', '#0d366b']
MODEL_ORDER = ['yolov8s', 'yolo11s', 'yolo12s', 'yolo26s', 'rtdetr-l', 'yolov8n', 'yolo11n', 'yolo12n']
DS_LABEL = {'pk': 'Pakistan', 'ad': 'AlphaDent', 'dai': 'DentalAI'}

def style():
    plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9, 'axes.edgecolor': INK2, 'axes.labelcolor': INK, 'xtick.color': INK2, 'ytick.color': INK2, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False, 'lines.linewidth': 2.0, 'lines.markersize': 6, 'figure.facecolor': 'white', 'axes.facecolor': 'white', 'figure.constrained_layout.use': True, 'axes.axisbelow': True})

def model_color(m: str, models) -> str:
    order = [x for x in MODEL_ORDER if x in models] + sorted((x for x in models if x not in MODEL_ORDER))
    return SERIES[order.index(m) % len(SERIES)]

def model_marker(m: str, models) -> str:
    order = [x for x in MODEL_ORDER if x in models] + sorted((x for x in models if x not in MODEL_ORDER))
    return MARKERS[order.index(m) % len(MARKERS)]

def save(fig, path_no_ext: Path):
    path_no_ext = Path(path_no_ext)
    path_no_ext.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path_no_ext.with_suffix('.png'), bbox_inches='tight')
    fig.savefig(path_no_ext.with_suffix('.pdf'), bbox_inches='tight')
    plt.close(fig)

In [ ]:
%%writefile /tmp/dentx_src/dentx/harmonize.py
from __future__ import annotations
import numpy as np
from PIL import Image, ImageDraw

def poly_to_box(xy: np.ndarray) -> np.ndarray:
    xy = np.asarray(xy, dtype=float).reshape(-1, 2)
    return np.array([xy[:, 0].min(), xy[:, 1].min(), xy[:, 0].max(), xy[:, 1].max()])

def check_and_clip(boxes: np.ndarray, tol: float):
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)
    if len(boxes) == 0:
        return (boxes, 0, 0, np.zeros(0, dtype=bool))
    oob = (boxes[:, 0] < -tol) | (boxes[:, 1] < -tol) | (boxes[:, 2] > 1 + tol) | (boxes[:, 3] > 1 + tol)
    outside = (boxes[:, 0] < 0) | (boxes[:, 1] < 0) | (boxes[:, 2] > 1) | (boxes[:, 3] > 1)
    clipped = np.clip(boxes, 0.0, 1.0)
    keep = (clipped[:, 2] - clipped[:, 0] > 0.0001) & (clipped[:, 3] - clipped[:, 1] > 0.0001)
    n_clipped = int((outside & ~oob & keep).sum())
    return (clipped[keep], int(oob.sum()), n_clipped, keep)

def rasterize(polys, size: int) -> np.ndarray:
    img = Image.new('1', (size, size), 0)
    d = ImageDraw.Draw(img)
    for p in polys:
        pts = [(float(x) * size, float(y) * size) for x, y in np.asarray(p).reshape(-1, 2)]
        if len(pts) >= 3:
            d.polygon(pts, fill=1)
        elif len(pts) == 2:
            d.rectangle([pts[0], pts[1]], fill=1)
    return np.array(img, dtype=bool)

def lesions_to_teeth(lesion_polys, tooth_polys, res: int=256, min_overlap: float=0.1):
    tooth_masks = [rasterize([t], res) for t in tooth_polys]
    tooth_boxes = [poly_to_box(t) for t in tooth_polys]
    chosen, extra = (set(), [])
    for lp in lesion_polys:
        lm = rasterize([lp], res)
        area = lm.sum()
        if area == 0:
            extra.append(poly_to_box(lp))
            continue
        best, best_ov = (-1, 0.0)
        for i, tm in enumerate(tooth_masks):
            ov = (lm & tm).sum() / area
            if ov > best_ov:
                best, best_ov = (i, ov)
        if best >= 0 and best_ov >= min_overlap:
            chosen.add(best)
        else:
            extra.append(poly_to_box(lp))
    boxes = [tooth_boxes[i] for i in sorted(chosen)] + extra
    return (np.clip(np.array(boxes), 0, 1) if boxes else np.zeros((0, 4)), len(extra))

def dhash(img: Image.Image, size: int=16) -> str:
    g = img.convert('L').resize((size + 1, size), Image.BILINEAR)
    a = np.asarray(g, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return np.packbits(bits).tobytes().hex()

def hamming_pairs(hashes, max_dist: int, chunk: int=256):
    if len(hashes) == 0:
        return []
    h = np.stack([np.frombuffer(bytes.fromhex(x), dtype=np.uint8) for x in hashes])
    bits = np.unpackbits(h, axis=1).astype(np.float32)
    n = len(bits)
    out = []
    for s in range(0, n, chunk):
        b = bits[s:s + chunk]
        d = b.sum(1)[:, None] + bits.sum(1)[None, :] - 2 * (b @ bits.T)
        ii, jj = np.nonzero(d <= max_dist + 0.5)
        ii = ii + s
        m = ii < jj
        out.extend(zip(ii[m].tolist(), jj[m].tolist()))
    return out

def to_yolo_lines(boxes: np.ndarray, cls: int=0) -> str:
    lines = []
    for x1, y1, x2, y2 in np.asarray(boxes).reshape(-1, 4):
        lines.append(f'{cls} {(x1 + x2) / 2:.6f} {(y1 + y2) / 2:.6f} {x2 - x1:.6f} {y2 - y1:.6f}')
    return '\n'.join(lines) + ('\n' if lines else '')

def read_yolo_boxes(path) -> np.ndarray:
    try:
        txt = open(path, encoding='utf-8').read().split('\n')
    except FileNotFoundError:
        return np.zeros((0, 4))
    rows = []
    for ln in txt:
        v = ln.split()
        if len(v) >= 5:
            cx, cy, w, h = map(float, v[1:5])
            rows.append([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])
    return np.array(rows) if rows else np.zeros((0, 4))

In [ ]:
%%writefile /tmp/dentx_src/dentx/infer.py
from __future__ import annotations
import argparse
import time
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from .config import Paths, load_resolved
from .corrupt import corrupt, parse
from .train import disable_integrations, is_done, load_model
from .utils import LOG, jdump, jload, setup_logging

def eval_sets_for(spec: dict, cfg: dict, avail) -> list:
    out = []
    if spec['kind'] == 'fewshot':
        return [dict(name=f"{spec['target']}_test", flip=False, feats=False, cam=False)]
    img = spec.get('split_col') == 'split_img'
    single = spec['cfg'] in cfg['corruptions']['sources'] and (not spec.get('filter')) and (not img)
    feats = bool(cfg['infer']['features']) and single
    flip = bool(cfg['infer']['flip']) and single
    cam = spec['cfg'] == 'pk' and spec['model'] in cfg['cam']['models']
    for ds in spec['datasets']:
        out.append(dict(name=f"{ds}{('img' if img else '')}_val", flip=flip, feats=feats, cam=False))
    for ds in sorted(avail):
        out.append(dict(name=f'{ds}_test', flip=flip, feats=feats, cam=cam and ds == 'pk'))
    if img:
        out.append(dict(name='pkimg_test', flip=False, feats=False, cam=False))
    if single:
        for ds in sorted(avail):
            for c in cfg['corruptions']['types']:
                for s in cfg['corruptions']['severities']:
                    out.append(dict(name=f'{ds}_test@{c}{s}', flip=flip and cfg['infer'].get('flip_corrupted', True), feats=feats, cam=False))
    fs = cfg['fewshot']
    if spec['model'] in fs['models'] and spec['cfg'] == fs['source']:
        for tgt in fs['targets']:
            if tgt in avail:
                out.append(dict(name=f'{tgt}_pool', flip=False, feats=False, cam=False))
    return out

def set_ids(name: str, man: pd.DataFrame) -> list:
    base, _, _ = parse(name)
    ds_part, which = base.rsplit('_', 1)
    img = ds_part.endswith('img')
    ds = ds_part[:-3] if img else ds_part
    m = man[(man.dataset == ds) & (man.usable == 1)]
    if which == 'pool':
        m = m[m.pool == 1]
    else:
        m = m[m['split_img' if img else 'split'] == which]
    return sorted(m.image_id)

def dump_path(paths: Paths, run_id: str, name: str, kind: str='json.gz') -> Path:
    return paths.dumps / run_id / f"{name.replace('@', '__')}.{kind}"

def missing_sets(paths: Paths, run_id: str, sets: list) -> list:
    out = []
    for s in sets:
        if not dump_path(paths, run_id, s['name']).exists():
            out.append(s)
        elif s['feats'] and (not dump_path(paths, run_id, s['name'], 'feat.npy').exists()):
            out.append(s)
        elif s['cam'] and (not dump_path(paths, run_id, s['name'], 'cam.json.gz').exists()):
            out.append(s)
    return out

class ImageCache:

    def __init__(self, man: pd.DataFrame, imgsz: int):
        self.path = dict(zip(man.image_id, man.image))
        self.imgsz = imgsz
        self.cache = {}

    def clean(self, iid: str) -> np.ndarray:
        if iid not in self.cache:
            im = cv2.imread(self.path[iid], cv2.IMREAD_COLOR)[:, :, ::-1]
            s = self.imgsz / max(im.shape[:2])
            if s < 1:
                im = cv2.resize(im, (max(1, round(im.shape[1] * s)), max(1, round(im.shape[0] * s))), interpolation=cv2.INTER_AREA)
            self.cache[iid] = np.ascontiguousarray(im)
        return self.cache[iid]

    def get(self, iid: str, kind: str, sev: int) -> np.ndarray:
        return corrupt(self.clean(iid), kind, sev, key=iid)

def _predict(model, arrays, cfg, device):
    I = cfg['infer']
    res_all = []
    half = device not in ('cpu',)
    for i in range(0, len(arrays), I['batch']):
        batch = [np.ascontiguousarray(a[:, :, ::-1]) for a in arrays[i:i + I['batch']]]
        res = model.predict(source=batch, imgsz=cfg['train']['imgsz'], conf=I['conf'], iou=I['nms_iou'], max_det=I['max_det'], half=half, device=device, verbose=False)
        for r in res:
            b = r.boxes
            if b is None or len(b) == 0:
                res_all.append([])
                continue
            xy = b.xyxyn.cpu().numpy()
            cf = b.conf.cpu().numpy()
            res_all.append(np.round(np.concatenate([xy, cf[:, None]], 1), 4).tolist())
    return res_all

def _unflip(preds):
    out = []
    for p in preds:
        out.append([[round(1 - x2, 4), y1, round(1 - x1, 4), y2, c] for x1, y1, x2, y2, c in p])
    return out

class FeatureNet:

    def __init__(self, model_name, weights, layer, device):
        import torch
        self.torch = torch
        self.dev = torch.device('cuda:0' if device not in ('cpu',) else 'cpu')
        self.net = load_model(model_name, weights).model.float().eval().to(self.dev)
        self.act = None
        mod = self.net.model[layer]
        mod.register_forward_hook(self._hook)

    def _hook(self, m, i, o):
        if isinstance(o, (list, tuple)):
            o = next((x for x in o if hasattr(x, 'dim') and x.dim() == 4), o[0])
        self.act = o.detach()

    def run(self, arrays, imgsz, batch, want_cam=False):
        torch = self.torch
        feats, cams = ([], [])
        for i in range(0, len(arrays), batch):
            x = np.stack([cv2.resize(a, (imgsz, imgsz), interpolation=cv2.INTER_AREA) for a in arrays[i:i + batch]])
            x = torch.from_numpy(x).permute(0, 3, 1, 2).float().div(255).to(self.dev)
            with torch.no_grad():
                self.net(x)
            a = self.act.float()
            feats.append(a.mean(dim=(2, 3)).cpu().numpy().astype(np.float16))
            if want_cam:
                cams.extend(eigencam(a.cpu().numpy()))
        return (np.concatenate(feats) if feats else np.zeros((0, 1), np.float16), cams)

def eigencam(act: np.ndarray) -> list:
    out = []
    for A in act:
        C, h, w = A.shape
        X = A.reshape(C, -1).T
        X = X - X.mean(0)
        try:
            _, _, vt = np.linalg.svd(X, full_matrices=False)
            proj = X @ vt[0]
        except np.linalg.LinAlgError:
            proj = np.zeros(h * w)
        if proj.max() < -proj.min():
            proj = -proj
        cam = np.maximum(proj, 0).reshape(h, w)
        out.append(cam / (cam.max() + 1e-09))
    return out

def eibr(cam: np.ndarray, boxes) -> float:
    H, W = (128, 128)
    c = cv2.resize(cam.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)
    m = np.zeros((H, W), bool)
    for x1, y1, x2, y2 in boxes:
        m[int(y1 * H):max(int(y1 * H) + 1, int(np.ceil(y2 * H))), int(x1 * W):max(int(x1 * W) + 1, int(np.ceil(x2 * W)))] = True
    tot = c.sum()
    return float(c[m].sum() / tot) if tot > 0 else float('nan')

def infer_run(work: str, run_id: str, device: str, deadline: float) -> None:
    cfg = load_resolved(work)
    paths = Paths(cfg)
    setup_logging(paths.logs, f'infer_{run_id}')
    disable_integrations()
    spec = jload(paths.plan)[run_id]
    man = pd.read_csv(paths.manifest, dtype={'group': str})
    avail = sorted(man.dataset.unique())
    sets = missing_sets(paths, run_id, eval_sets_for(spec, cfg, avail))
    if not sets:
        return
    weights = str(paths.runs / run_id / 'best.pt')
    model = load_model(spec['model'], weights)
    fnet = None
    imgsz = cfg['train']['imgsz']
    cache = ImageCache(man, imgsz)
    gts = {}
    for s in sets:
        if time.time() > deadline:
            LOG.warning('deadline reached; %s left unfinished', run_id)
            return
        t0 = time.time()
        name = s['name']
        base, kind, sev = parse(name)
        ids = set_ids(name, man)
        arrays = [cache.get(i, kind, sev) for i in ids]
        dp = dump_path(paths, run_id, name)
        if not dp.exists():
            d = dict(run=run_id, set=name, ids=ids, p=_predict(model, arrays, cfg, device))
            if s['flip']:
                d['pf'] = _unflip(_predict(model, [a[:, ::-1] for a in arrays], cfg, device))
            jdump(d, dp)
        if s['feats'] or s['cam']:
            if fnet is None:
                fnet = FeatureNet(spec['model'], weights, cfg['infer']['feature_layer'], device)
            feats, cams = fnet.run(arrays, imgsz, cfg['infer']['batch'], want_cam=s['cam'])
            if s['feats']:
                np.save(dump_path(paths, run_id, name, 'feat.npy'), feats)
            if s['cam']:
                ds = base.split('_')[0]
                if ds not in gts:
                    gts[ds] = jload(paths.gt / f'{ds}.json.gz')
                vals, ex = ({}, {})
                for iid, cam in zip(ids, cams):
                    b = gts[ds][iid]['boxes']
                    if b:
                        vals[iid] = eibr(cam, b)
                        if len(ex) < 4 * cfg['cam']['n_examples']:
                            ex[iid] = np.round(cv2.resize(cam, (48, 48)), 3).tolist()
                jdump(dict(eibr=vals, examples=ex), dump_path(paths, run_id, name, 'cam.json.gz'))
        LOG.info('%s %s: %d images in %.0f s', run_id, name, len(ids), time.time() - t0)
    LOG.info('inference done for %s', run_id)

def run_inference(cfg: dict, paths: Paths, plan: dict, deadline: float, hard_deadline: float) -> None:
    from .train import gpu_slots, run_queue
    man = pd.read_csv(paths.manifest, dtype={'group': str})
    avail = sorted(man.dataset.unique())
    n_by_set = {}
    items = []
    per_img = 0.3 if gpu_slots(cfg) == ['cpu'] else 0.02
    for rid, spec in sorted(plan.items(), key=lambda kv: (kv[1]['kind'] != 'main', kv[1]['priority'])):
        if not is_done(paths, rid):
            continue
        miss = missing_sets(paths, rid, eval_sets_for(spec, cfg, avail))
        if not miss:
            continue
        n = 0
        for s in miss:
            if s['name'] not in n_by_set:
                n_by_set[s['name']] = len(set_ids(s['name'], man))
            n += n_by_set[s['name']] * (1 + s['flip'] + s['feats'])
        items.append((rid, 30 + n * per_img, [], ['--run', rid, '--deadline', str(hard_deadline)]))
    if not items:
        LOG.info('all inference dumps present')
        return
    LOG.info('inference queue: %d runs', len(items))
    run_queue(cfg, paths, plan, deadline, hard_deadline, 'dentx.infer', items, 'infer')

def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--work', required=True)
    ap.add_argument('--run', required=True)
    ap.add_argument('--device', default='0')
    ap.add_argument('--deadline', type=float, default=float('inf'))
    a = ap.parse_args(argv)
    infer_run(a.work, a.run, a.device, a.deadline)
if __name__ == '__main__':
    main()

In [ ]:
%%writefile /tmp/dentx_src/dentx/metrics.py
from __future__ import annotations
import numpy as np
GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)

def iou_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0])
    iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2])
    iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.maximum(aa[:, None] + ab[None, :] - inter, 1e-12)

def iomin_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0])
    iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2])
    iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.maximum(np.minimum(aa[:, None], ab[None, :]), 1e-12)

def greedy_match(sim: np.ndarray, thr: float):
    P, G = sim.shape
    tp = np.zeros(P, bool)
    used = np.zeros(G, bool)
    if G == 0:
        return (tp, used)
    for i in range(P):
        s = np.where(used, -1.0, sim[i])
        j = int(np.argmax(s))
        if s[j] >= thr:
            tp[i] = True
            used[j] = True
    return (tp, used)

class SetEval:

    def __init__(self, ids, preds, gts, groups=None, iou: float=0.5, iomin: float=0.5):
        self.ids = list(ids)
        n = len(self.ids)
        self.n = n
        self.groups = np.asarray(groups if groups is not None else self.ids)
        pi, pc, ptp, phit, gi, gmax = ([], [], [], [], [], [])
        self.ious = []
        self.n_gt_img = np.zeros(n, int)
        self.score = np.zeros(n)
        self.pos = np.zeros(n, bool)
        for k, (p, g) in enumerate(zip(preds, gts)):
            p = np.asarray(p, dtype=float).reshape(-1, 5)
            g = np.asarray(g, dtype=float).reshape(-1, 4)
            if len(p):
                p = p[np.argsort(-p[:, 4], kind='stable')]
            M = iou_matrix(p[:, :4], g)
            Mm = iomin_matrix(p[:, :4], g)
            tp, _ = greedy_match(M, iou)
            hit = (Mm >= iomin).any(1) if len(g) else np.zeros(len(p), bool)
            if len(g):
                hm = np.where(Mm >= iomin, p[:, 4:5], -1.0)
                gm = hm.max(0) if len(p) else np.full(len(g), -1.0)
            else:
                gm = np.zeros(0)
            pi.append(np.full(len(p), k))
            pc.append(p[:, 4])
            ptp.append(tp)
            phit.append(hit)
            gi.append(np.full(len(g), k))
            gmax.append(gm)
            self.ious.append((p[:, 4], M))
            self.n_gt_img[k] = len(g)
            self.score[k] = p[:, 4].max() if len(p) else 0.0
            self.pos[k] = len(g) > 0
        cat = lambda xs, dt: np.concatenate(xs).astype(dt) if xs else np.zeros(0, dt)
        self.p_img, self.p_conf = (cat(pi, int), cat(pc, float))
        self.p_tp, self.p_hit = (cat(ptp, bool), cat(phit, bool))
        self.g_img, self.g_max = (cat(gi, int), cat(gmax, float))
        self.n_gt = int(self.n_gt_img.sum())

    def counts(self, thr: float, level: str='strict') -> np.ndarray:
        n = self.n
        m = self.p_conf >= thr
        if level == 'image':
            yes = self.score >= thr
            return np.stack([yes & self.pos, yes & ~self.pos, yes & self.pos, ~yes & self.pos, ~yes & ~self.pos], 1).astype(float)
        flag = self.p_tp if level == 'strict' else self.p_hit
        ph = np.bincount(self.p_img[m], weights=flag[m], minlength=n)
        pm = np.bincount(self.p_img[m], weights=~flag[m], minlength=n)
        if level == 'strict':
            gh = ph
        else:
            gh = np.bincount(self.g_img, weights=self.g_max >= thr, minlength=n)
        return np.stack([ph, pm, gh, self.n_gt_img - gh], 1)

    def prf(self, thr: float, level: str='strict') -> dict:
        return prf_from_sums(self.counts(thr, level).sum(0), level)

    def best_threshold(self, level: str='strict') -> float:
        f1 = [self.prf(t, level)['F1'] for t in GRID]
        return float(GRID[int(np.nanargmax(f1))]) if np.isfinite(np.nanmax(f1)) else 0.25

    def ap(self, thrs=(0.5,)) -> list:
        if self.n_gt == 0:
            return [np.nan] * len(thrs)
        out = []
        for t in thrs:
            confs, tps = ([], [])
            for conf, M in self.ious:
                if len(conf) == 0:
                    continue
                tp, _ = greedy_match(M, t) if M.shape[1] else (np.zeros(len(conf), bool), None)
                confs.append(conf)
                tps.append(tp)
            if not confs:
                out.append(0.0)
                continue
            conf = np.concatenate(confs)
            tp = np.concatenate(tps)[np.argsort(-conf, kind='stable')]
            ctp, cfp = (np.cumsum(tp), np.cumsum(~tp))
            rec = ctp / self.n_gt
            prec = ctp / np.maximum(ctp + cfp, 1e-12)
            env = np.maximum.accumulate(prec[::-1])[::-1]
            rs = np.linspace(0, 1, 101)
            idx = np.searchsorted(rec, rs, side='left')
            q = np.where(idx < len(env), env[np.minimum(idx, len(env) - 1)], 0.0)
            out.append(float(q.mean()))
        return out

    def auroc(self, weights=None) -> float:
        return weighted_auroc(self.score, self.pos, weights)

    def ece(self, thr: float, bins: int=10):
        m = self.p_conf >= thr
        conf, ok = (self.p_conf[m], self.p_tp[m])
        if len(conf) == 0:
            return (np.nan, [])
        edges = np.linspace(0, 1, bins + 1)
        b = np.clip(np.digitize(conf, edges[1:-1]), 0, bins - 1)
        e, rows = (0.0, [])
        for k in range(bins):
            s = b == k
            if s.any():
                e += s.mean() * abs(ok[s].mean() - conf[s].mean())
                rows.append((float(edges[k]), float(edges[k + 1]), int(s.sum()), float(conf[s].mean()), float(ok[s].mean())))
        return (float(e), rows)

def prf_from_sums(s: np.ndarray, level: str='strict') -> dict:
    s = np.asarray(s, float)
    ph, pm, gh, gm = (s[..., 0], s[..., 1], s[..., 2], s[..., 3])
    with np.errstate(invalid='ignore', divide='ignore'):
        P = ph / (ph + pm)
        R = gh / (gh + gm)
        F1 = 2 * P * R / (P + R)
        F1 = np.where(P + R == 0, 0.0, F1)
        out = dict(P=P, R=R, F1=F1)
        if level == 'image' and s.shape[-1] > 4:
            out['spec'] = s[..., 4] / (s[..., 4] + pm)
    return {k: float(v) if np.ndim(v) == 0 else v for k, v in out.items()}

def weighted_auroc(score, pos, w=None) -> float:
    score, pos = (np.asarray(score, float), np.asarray(pos, bool))
    w = np.ones(len(score)) if w is None else np.asarray(w, float)
    wp, wn = (w[pos].sum(), w[~pos].sum())
    if wp == 0 or wn == 0:
        return np.nan
    u, r = np.unique(score, return_inverse=True)
    negw = np.bincount(r, weights=w * ~pos, minlength=len(u))
    below = np.cumsum(negw) - negw
    return float((w[pos] * (below[r[pos]] + 0.5 * negw[r[pos]])).sum() / (wp * wn))

class Bootstrap:

    def __init__(self, groups, B: int, seed: int):
        u, self.inv = np.unique(np.asarray(groups).astype(str), return_inverse=True)
        G = len(u)
        rng = np.random.default_rng(seed)
        idx = rng.integers(0, G, size=(B, G))
        self.W = np.zeros((B, G), np.float32)
        np.add.at(self.W, (np.repeat(np.arange(B), G), idx.ravel()), 1.0)
        self.B, self.G = (B, G)

    def sums(self, per_image: np.ndarray) -> np.ndarray:
        S = np.zeros((self.G, per_image.shape[1]))
        np.add.at(S, self.inv, per_image)
        return self.W @ S

    def image_weights(self, b: int) -> np.ndarray:
        return self.W[b, self.inv]

def ci(x, alpha: float=0.05):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return (np.nan, np.nan)
    return (float(np.quantile(x, alpha / 2)), float(np.quantile(x, 1 - alpha / 2)))

def evaluate(se: SetEval, thr: float, boot: Bootstrap | None, levels=('strict', 'lenient', 'image'), want_ap: bool=True, ap5095: bool=True, ece_bins: int=10, auroc_boot: int=500) -> tuple:
    out = dict(n_img=se.n, n_pos=int(se.pos.sum()), n_gt=se.n_gt, thr=thr)
    reps = {}
    for lv in levels:
        c = se.counts(thr, lv)
        pt = prf_from_sums(c.sum(0), lv)
        for k, v in pt.items():
            out[f'{lv}_{k}'] = v
        if boot is not None:
            r = prf_from_sums(boot.sums(c), lv)
            reps[lv] = r['F1']
            for k in ('P', 'R', 'F1') + (('spec',) if lv == 'image' else ()):
                lo, hi = ci(r[k])
                out[f'{lv}_{k}_lo'], out[f'{lv}_{k}_hi'] = (lo, hi)
    out['img_auc'] = se.auroc()
    if boot is not None and auroc_boot:
        a = [se.auroc(boot.image_weights(b)) for b in range(min(auroc_boot, boot.B))]
        out['img_auc_lo'], out['img_auc_hi'] = ci(a)
        reps['auc'] = np.array(a)
    if want_ap:
        thrs = np.round(np.arange(0.5, 0.951, 0.05), 2) if ap5095 else (0.5,)
        aps = se.ap(thrs)
        out['AP50'] = aps[0]
        out['AP50_95'] = float(np.nanmean(aps)) if ap5095 else np.nan
    out['ece'], _ = se.ece(thr, ece_bins)
    out['n_pred_thr'] = int((se.p_conf >= thr).sum())
    return (out, reps)

In [ ]:
%%writefile /tmp/dentx_src/dentx/package.py
from __future__ import annotations
import shutil
import time
import zipfile
from pathlib import Path
from .config import Paths
from .train import is_done
from .utils import LOG
RUN_FILES = ('done.json', 'results.csv', 'args.yaml', 'results.png')
CACHE_NAME = 'dentx_data_cache'
GT_FILES = ('{ds}.json.gz', '{ds}_summary.json', '{ds}_near_duplicates.csv')
TOP_FILES = ('manifest.csv', 'plan.json', 'config_resolved.yaml')

def make_zip(cfg: dict, paths: Paths, plan: dict) -> Path:
    ts = time.strftime('%Y%m%d_%H%M%S')
    dest = paths.work.parent / f"dentx_results_{cfg['profile']}_{ts}.zip"
    inc_w = cfg['package']['include_weights']
    with zipfile.ZipFile(dest, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for f in TOP_FILES:
            if (paths.work / f).exists():
                z.write(paths.work / f, f'dentx_run/{f}')
        for sub in ('results', 'dumps', 'gt', 'logs'):
            d = paths.work / sub
            if d.exists():
                for p in sorted(d.rglob('*')):
                    if p.is_file():
                        z.write(p, f'dentx_run/{p.relative_to(paths.work).as_posix()}')
        for rid in plan:
            d = paths.runs / rid
            for f in RUN_FILES + (('best.pt',) if inc_w else ()):
                if (d / f).exists():
                    z.write(d / f, f'dentx_run/runs/{rid}/{f}', compress_type=zipfile.ZIP_STORED if f.endswith('.pt') else None)
    missing = verify_zip(dest, plan, paths)
    LOG.info('results zip: %s (%.1f MB)%s', dest, dest.stat().st_size / 2 ** 20, f'; {len(missing)} expected item(s) missing: {missing[:5]}' if missing else '; all expected items present')
    return dest

def verify_zip(zpath: Path, plan: dict, paths: Paths) -> list:
    with zipfile.ZipFile(zpath) as z:
        names = set(z.namelist())
    need = [f'dentx_run/{f}' for f in TOP_FILES] + ['dentx_run/results/RESULTS.md', 'dentx_run/results/env.json', 'dentx_run/results/tables/table1_datasets.csv', 'dentx_run/results/tables/all_metrics.csv']
    need += [f'dentx_run/runs/{r}/done.json' for r in plan if is_done(paths, r)]
    return [n for n in need if n not in names]

def restore_state(paths: Paths) -> int:
    n = 0
    for base in paths.inputs:
        base = Path(base)
        if not base.exists():
            continue
        for d in base.rglob('dentx_run'):
            if d.is_dir() and (d / 'runs').is_dir() and (d.resolve() != paths.work.resolve()):
                n += _copy_tree(d, paths)
        for zp in sorted(base.rglob('dentx_results_*.zip')):
            with zipfile.ZipFile(zp) as z:
                for name in z.namelist():
                    if not name.startswith('dentx_run/') or name.endswith('/'):
                        continue
                    rel = name[len('dentx_run/'):]
                    if not rel.startswith(('runs/', 'dumps/', 'gt/', 'manifest.csv')):
                        continue
                    dst = paths.work / rel
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True)
                        with z.open(name) as src, open(dst, 'wb') as out:
                            shutil.copyfileobj(src, out)
                        n += 1
    if n:
        LOG.info('restored %d file(s) from previous session(s)', n)
    return n

def _copy_tree(src: Path, paths: Paths) -> int:
    n = 0
    for sub in ('runs', 'dumps', 'gt'):
        for p in (src / sub).rglob('*') if (src / sub).exists() else []:
            if p.is_file():
                dst = paths.work / p.relative_to(src)
                if not dst.exists():
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(p, dst)
                    n += 1
    if (src / 'manifest.csv').exists() and (not paths.manifest.exists()):
        shutil.copy2(src / 'manifest.csv', paths.manifest)
    return n

def save_data_cache(paths: Paths, datasets) -> None:
    root = paths.work.parent / CACHE_NAME
    for ds in datasets:
        src = paths.data / ds
        dst = root / ds
        if not (src / '.done.json').exists():
            continue
        if (dst / '.done.json').exists() and (dst / '.done.json').read_text() == (src / '.done.json').read_text():
            continue
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        (dst / '_gt').mkdir(exist_ok=True)
        for f in GT_FILES:
            g = paths.gt / f.format(ds=ds)
            if g.exists():
                shutil.copy2(g, dst / '_gt' / g.name)
        LOG.info('cached prepared %s for later sessions (%s)', ds, dst)

def restore_data_cache(paths: Paths, cfg_hash: str) -> list:
    import json
    restored = []
    own = paths.work.parent / CACHE_NAME
    cands = [own] if own.is_dir() else []
    for base in paths.inputs:
        if Path(base).exists():
            cands += [d for d in Path(base).rglob(CACHE_NAME) if d.is_dir()]
    for root in cands:
        for d in sorted(root.iterdir()):
            ds = d.name
            mk = d / '.done.json'
            if ds in restored or not mk.exists():
                continue
            if json.loads(mk.read_text()).get('hash') != cfg_hash:
                LOG.info('cached %s was prepared with other settings; ignoring it', ds)
                continue
            dst = paths.data / ds
            if not ((dst / '.done.json').exists() and (dst / '.done.json').read_text() == mk.read_text()):
                if dst.exists():
                    shutil.rmtree(dst)
                shutil.copytree(d, dst, ignore=shutil.ignore_patterns('_gt'))
                rec = dst / 'records.csv'
                if rec.exists():
                    import pandas as pd
                    df = pd.read_csv(rec, dtype={'group': str})
                    df['image'] = [str(dst / 'images' / Path(x).name) for x in df['image']]
                    df.to_csv(rec, index=False)
            for g in (d / '_gt').glob('*') if (d / '_gt').exists() else []:
                if not (paths.gt / g.name).exists():
                    paths.gt.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(g, paths.gt / g.name)
            restored.append(ds)
    if restored:
        LOG.info('prepared data restored from cache: %s', restored)
    return restored

In [ ]:
%%writefile /tmp/dentx_src/dentx/pipeline.py
from __future__ import annotations
import argparse
import time
import traceback
import yaml
from . import download, prepare, report
from .analysis import Store, compute_master, save_master
from .config import Paths, load_config, save_resolved
from .infer import run_inference
from .config import config_hash, on_kaggle
from .package import make_zip, restore_data_cache, restore_state, save_data_cache
from .train import build_cfg_dirs, build_plan, drop_stale, run_training
from .utils import LOG, jdump, setup_logging
EXPERIMENTS = ['e0_replicate', 'e1_indomain', 'e2_lodo', 'e3_protocol', 'e4_labelfree', 'e5_fewshot']

def parse_overrides(items):
    out = {}
    for it in items or []:
        k, v = it.split('=', 1)
        d = out
        keys = k.split('.')
        for kk in keys[:-1]:
            d = d.setdefault(kk, {})
        d[keys[-1]] = yaml.safe_load(v)
    return out

def analyze(cfg, paths, plan) -> tuple[dict, dict]:
    import importlib
    store = Store(cfg, paths)
    master, reps = compute_master(store)
    save_master(master, reps, paths)
    summaries, errors = ({}, {})
    for name in EXPERIMENTS:
        try:
            mod = importlib.import_module(f'dentx.experiments.{name}')
            summaries[name] = mod.run(store, master, reps, paths)
            LOG.info('analysis %s done', name)
        except Exception as e:
            errors[name] = f'{type(e).__name__}: {e}'
            LOG.error('analysis %s failed:\n%s', name, traceback.format_exc())
    jdump(summaries, paths.results / 'summaries.json')
    return (summaries, errors)

def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument('--profile', default='full')
    ap.add_argument('--work', default=None)
    ap.add_argument('--synthetic', action='store_true', help='use generated mini-datasets instead of downloads')
    ap.add_argument('--stage', default='all', choices=['all', 'prepare', 'train', 'infer', 'analyze'])
    ap.add_argument('--hours', type=float, default=None, help='session budget (overrides budget.hours)')
    ap.add_argument('--set', nargs='*', default=[], help='config overrides, e.g. train.epochs=50')
    a = ap.parse_args(argv)
    t0 = time.time()
    ov = parse_overrides(a.set)
    if a.hours:
        ov.setdefault('budget', {})['hours'] = a.hours
    cfg = load_config(a.profile, ov, work=a.work)
    paths = Paths(cfg).make()
    setup_logging(paths.logs, 'pipeline')
    save_resolved(cfg, paths)
    report.env_record(cfg, paths)
    B = cfg['budget']
    t_end = t0 + B['hours'] * 3600
    infer_deadline = t_end - B['analysis_reserve_h'] * 3600
    train_deadline = infer_deadline - B['infer_reserve_h'] * 3600
    LOG.info('profile=%s work=%s budget=%.1f h', cfg['profile'], paths.work, B['hours'])
    plan, summaries, errors = ({}, {}, {})
    try:
        restore_state(paths)
        if a.synthetic:
            from .synthetic import make_all
            if not any(paths.raw.glob('*synth*')):
                make_all(paths.raw)
        restore_data_cache(paths, config_hash(cfg))
        sources = {ds: paths.data / ds if prepare.is_prepared(ds, cfg, paths) else download.ensure_dataset(ds, cfg, paths) for ds in ('pk', 'ad', 'dai')}
        if not any(sources.values()):
            raise RuntimeError('no dataset available')
        man = prepare.prepare_all(cfg, paths, sources)
        if on_kaggle() and cfg['package'].get('cache_prepared_data', True):
            save_data_cache(paths, [ds for ds, src in sources.items() if src is not None])
        if cfg['prepare'].get('cleanup_raw'):
            import shutil
            shutil.rmtree(paths.raw, ignore_errors=True)
        report.dataset_table(paths)
        plan = build_plan(cfg, man)
        jdump(plan, paths.plan)
        drop_stale(plan, paths)
        build_cfg_dirs(plan, man, paths)
        if a.stage in ('all', 'train'):
            run_training(cfg, paths, plan, train_deadline, train_deadline + 0.5 * B['infer_reserve_h'] * 3600)
        if a.stage in ('all', 'infer'):
            run_inference(cfg, paths, plan, infer_deadline, infer_deadline)
        if a.stage in ('all', 'analyze', 'infer', 'train'):
            summaries, errors = analyze(cfg, paths, plan)
    except Exception as e:
        errors['pipeline'] = f'{type(e).__name__}: {e}'
        LOG.error('pipeline error:\n%s', traceback.format_exc())
    finally:
        try:
            if plan:
                report.run_table(paths, plan)
                report.write_results_md(cfg, paths, plan, summaries, errors)
            z = make_zip(cfg, paths, plan)
            print(f'\nRESULTS ZIP: {z}')
        except Exception:
            LOG.error('packaging failed:\n%s', traceback.format_exc())
    LOG.info('session finished in %.2f h', (time.time() - t0) / 3600)
if __name__ == '__main__':
    main()

In [ ]:
%%writefile /tmp/dentx_src/dentx/prepare.py
from __future__ import annotations
import io
import json
import re
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageOps
from .config import Paths, config_hash
from .harmonize import check_and_clip, dhash, hamming_pairs, lesions_to_teeth, poly_to_box, to_yolo_lines
from .utils import LOG, RawTree, jdump, jload
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

class UF:

    def __init__(self, items):
        self.p = {i: i for i in items}

    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

    def union(self, a, b):
        ra, rb = (self.find(a), self.find(b))
        if ra != rb:
            self.p[max(ra, rb)] = min(ra, rb)

def _sanitize(s: str) -> str:
    return re.sub('[^A-Za-z0-9_\\-]+', '_', s).strip('_')
PK_SESS = re.compile('anonymous[_-](\\d{3})[_-](\\d{3})[_-](\\d+)[_-](\\d+)_(\\d+)_', re.I)
PK_PILOT = re.compile('anonymous-([a-z]+?)view-(\\d+)', re.I)
PK_COND = {'retractors': 'retractor', 'no_retractors': 'no_retractor', 'pilot': 'pilot'}

def pk_tasks(tree: RawTree, pcfg: dict):
    names = tree.names
    lm = {}
    for n in names:
        m = re.search('Annotations/Labelme/([^/]+)/(?:[^/]+/)*([^/]+)\\.json$', n)
        if m:
            lm[m.group(1), m.group(2)] = n
    tasks = []
    for n in names:
        m = re.search('Dataset/Images/(retractors|no_retractors|pilot)/([^/]+)/(?:[^/]+/)*([^/]+)\\.(jpe?g|png)$', n, re.I)
        if not m:
            continue
        cond_dir, view, stem = (m.group(1), m.group(2), m.group(3))
        tasks.append(dict(raw=n, ds='pk', stem=stem, condition=PK_COND[cond_dir], view=view, ann=lm.get((cond_dir, stem))))
    _pk_groups(tasks, pcfg)
    return tasks

def _pk_groups(tasks, pcfg):
    sessions = {}
    pilot = []
    for t in tasks:
        m = PK_SESS.search(t['stem'])
        if m:
            key = '-'.join(m.groups()[:4])
            t['session'] = key
            sessions[key] = dict(t=int(m.group(5)) / 1000.0, cond=t['condition'], seq=int(m.group(3)))
            continue
        m = PK_PILOT.search(t['stem'])
        if m:
            pilot.append((int(m.group(2)) / 1000.0, m.group(1).lower(), t))
        else:
            t['session'] = 'solo_' + t['stem']
    uf = UF(list(sessions))
    byseq = defaultdict(list)
    for k, v in sessions.items():
        byseq[v['seq']].append(k)
    for k, v in sessions.items():
        for d in range(-pcfg['pk_link_seq_delta'], pcfg['pk_link_seq_delta'] + 1):
            for k2 in byseq.get(v['seq'] + d, []):
                v2 = sessions[k2]
                if v2['cond'] != v['cond'] and abs(v2['t'] - v['t']) <= pcfg['pk_link_window_s']:
                    uf.union(k, k2)
    pilot.sort(key=lambda x: x[0])
    sid, last_t, seen = (0, None, set())
    for ts, view, t in pilot:
        if last_t is None or view.startswith('frontal') or ts - last_t > pcfg['pilot_gap_s'] or (view in seen):
            sid += 1
            seen = set()
        seen.add(view)
        last_t = ts
        t['session'] = f'pilot{sid:04d}'
    for t in tasks:
        s = t['session']
        t['group'] = 'pk_' + (uf.find(s) if s in sessions else s)

def pk_labels(tree: RawTree, task: dict, W: int, H: int, tol: float):
    if not task['ann']:
        return dict(boxes=np.zeros((0, 4)), sub=[], n_oob=0, n_clipped=0, geom_note='')
    j = json.loads(tree.read(task['ann']).decode('utf-8', 'replace'))
    lw, lh = (j.get('imageWidth') or W, j.get('imageHeight') or H)
    note = ''
    if (lw, lh) != (W, H):
        note = 'rotated' if (lw, lh) == (H, W) else 'rescaled'
    boxes, sub = ([], [])
    for s in j.get('shapes', []):
        pts = np.asarray(s.get('points', []), dtype=float).reshape(-1, 2)
        if len(pts) < 2:
            continue
        b = poly_to_box(pts) / np.array([lw, lh, lw, lh])
        boxes.append(b)
        sub.append(str(s.get('label', '')).strip())
    boxes = np.array(boxes) if boxes else np.zeros((0, 4))
    kept, n_oob, n_clip, keep = check_and_clip(boxes, tol)
    if note == 'rotated':
        n_oob = max(n_oob, 1)
    return dict(boxes=kept, sub=[s for s, k in zip(sub, keep) if k], n_oob=n_oob, n_clipped=n_clip, geom_note=note)
AD_NAME = re.compile('(p\\d+)_([MFmf])_(\\d+)_', re.I)

def ad_tasks(tree: RawTree, pcfg: dict):
    names = tree.names
    yaml_name = next((n for n in names if n.endswith('yolo_seg_train.yaml')), None)
    prefix = yaml_name[:-len('yolo_seg_train.yaml')] if yaml_name else ''
    caries_ids, class_names = (set(range(3, 9)), {})
    if yaml_name:
        y = yaml.safe_load(tree.read(yaml_name))
        nm = y.get('names', {})
        class_names = dict(enumerate(nm)) if isinstance(nm, list) else {int(k): v for k, v in nm.items()}
        caries_ids = {k for k, v in class_names.items() if pcfg['ad_caries_name'] in str(v).lower()}
    labels = {n for n in names if n.startswith(prefix + 'labels/') and n.endswith('.txt')}
    tasks = []
    for n in names:
        m = re.match(re.escape(prefix) + 'images/(train|valid)/([^/]+)\\.(jpe?g|png)$', n, re.I)
        if not m:
            continue
        split, stem = (m.group(1), m.group(2))
        lab = f'{prefix}labels/{split}/{stem}.txt'
        pm = AD_NAME.search(stem)
        tasks.append(dict(raw=n, ds='ad', stem=stem, ann=lab if lab in labels else None, group='ad_' + (pm.group(1).lower() if pm else stem), sex=pm.group(2).upper() if pm else '', age=int(pm.group(3)) if pm else np.nan, caries_ids=caries_ids, class_names=class_names, official_split=split))
    return tasks

def ad_labels(tree: RawTree, task: dict, W: int, H: int, tol: float):
    boxes, sub, other = ([], [], Counter())
    if task['ann']:
        for ln in tree.read(task['ann']).decode('utf-8', 'replace').splitlines():
            v = ln.split()
            if len(v) < 5:
                continue
            c = int(float(v[0]))
            coords = np.array(v[1:], dtype=float)
            if c not in task['caries_ids']:
                other[task['class_names'].get(c, str(c))] += 1
                continue
            if len(coords) == 4:
                cx, cy, w, h = coords
                b = np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])
            else:
                b = poly_to_box(coords[:len(coords) // 2 * 2])
            boxes.append(b)
            sub.append(str(task['class_names'].get(c, c)))
    boxes = np.array(boxes) if boxes else np.zeros((0, 4))
    kept, n_oob, n_clip, keep = check_and_clip(boxes, tol)
    return dict(boxes=kept, sub=[s for s, k in zip(sub, keep) if k], n_oob=n_oob, n_clipped=n_clip, geom_note='', other=dict(other))

def _rf_base(stem: str) -> str:
    s = re.split('\\.rf\\.', stem)[0]
    s = re.sub('_(jpe?g|png|bmp)$', '', s, flags=re.I)
    return s.lower()

def _is(nm: str, keys) -> bool:
    nm = nm.lower()
    return any((nm == k or nm.startswith(k) for k in keys))

def dai_tasks(tree: RawTree, pcfg: dict):
    names = tree.names
    cocos = [n for n in names if n.endswith('_annotations.coco.json')]
    car = [c.lower() for c in pcfg['dai_caries_names']]
    tooth = pcfg['dai_tooth_name'].lower()
    tasks = []
    name_set = set(names)
    sly = [n for n in names if re.search('(^|/)(train|valid|test)/ann/[^/]+\\.json$', n)]
    csvs = [n for n in names if n.endswith('_classes.csv')]
    if sly:
        for an in sorted(sly, key=lambda n: ({'train': 0, 'valid': 1, 'test': 2}[re.search('(train|valid|test)/ann/', n).group(1)], n)):
            split = re.search('(train|valid|test)/ann/', an).group(1)
            raw = an.replace('/ann/', '/img/')[:-len('.json')]
            if raw not in name_set:
                continue
            d = json.loads(tree.read(an))
            w, h = (d['size']['width'], d['size']['height'])
            les, teeth, other = ([], [], Counter())
            for o in d.get('objects', []):
                pts = np.asarray(o.get('points', {}).get('exterior', []), dtype=float).reshape(-1, 2)
                if len(pts) < 2:
                    continue
                poly = pts / np.array([w, h])
                if len(pts) == 2:
                    (x1, y1), (x2, y2) = poly
                    poly = np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]])
                nm = str(o.get('classTitle', '')).lower()
                if _is(nm, car):
                    les.append((poly, nm))
                elif _is(nm, [tooth]):
                    teeth.append(poly)
                else:
                    other[nm] += 1
            stem = Path(raw).stem
            tasks.append(dict(raw=raw, ds='dai', stem=stem, base=_rf_base(stem), lesions=les, teeth=teeth, other=dict(other), official_split=split, fmt='polygons'))
    elif csvs:
        for cv in sorted(csvs):
            split_dir = cv[:-len('_classes.csv')]
            cmap = {}
            for ln in tree.read(cv).decode('utf-8', 'replace').splitlines():
                parts = [x.strip() for x in ln.split(',')]
                if len(parts) >= 2 and parts[0].lstrip('-').isdigit():
                    cmap[int(parts[0])] = parts[1].lower()
            car_v = [v for v, nm in cmap.items() if _is(nm, car)]
            tooth_v = [v for v, nm in cmap.items() if _is(nm, [tooth])]
            for n in names:
                if not n.startswith(split_dir) or '/' in n[len(split_dir):] or (not n.lower().endswith(IMG_EXT)):
                    continue
                if n.endswith('_mask.png'):
                    continue
                mask = n.rsplit('.', 1)[0] + '_mask.png'
                if mask not in name_set:
                    continue
                stem = Path(n).stem
                tasks.append(dict(raw=n, ds='dai', stem=stem, base=_rf_base(stem), mask=mask, car_v=car_v, tooth_v=tooth_v, lesions=None, teeth=None, other={}, official_split=split_dir.rstrip('/').split('/')[-1], fmt='semantic_mask'))
    elif cocos:
        for cj in sorted(cocos, key=lambda n: ({'train': 0, 'valid': 1, 'test': 2}.get(n.split('/')[-2], 3), n)):
            split_dir = cj[:-len('_annotations.coco.json')]
            d = json.loads(tree.read(cj))
            cats = {c['id']: c['name'].lower() for c in d.get('categories', [])}
            anns = defaultdict(list)
            for a in d.get('annotations', []):
                anns[a['image_id']].append(a)
            for im in d.get('images', []):
                raw = split_dir + im['file_name']
                if raw not in name_set:
                    continue
                les, teeth, other = ([], [], Counter())
                for a in anns[im['id']]:
                    nm = cats.get(a['category_id'], '')
                    polys = _coco_polys(a, im['width'], im['height'])
                    if any((nm == c or nm.startswith(c) for c in car)):
                        les += [(p, nm) for p in polys]
                    elif nm == tooth or nm.startswith(tooth):
                        teeth += polys
                    else:
                        other[nm] += 1
                stem = Path(im['file_name']).stem
                tasks.append(dict(raw=raw, ds='dai', stem=stem, base=_rf_base(stem), lesions=les, teeth=teeth, other=dict(other), official_split=split_dir.rstrip('/').split('/')[-1]))
    else:
        y = next((n for n in names if n.endswith(('data.yaml', 'dataset.yaml'))), None)
        if y is None:
            raise FileNotFoundError('DentalAI: neither COCO json nor data.yaml found')
        nm = yaml.safe_load(tree.read(y)).get('names', {})
        cls = dict(enumerate(nm)) if isinstance(nm, list) else {int(k): v for k, v in nm.items()}
        cls = {k: str(v).lower() for k, v in cls.items()}
        lab_set = set((n for n in names if n.endswith('.txt')))
        for n in names:
            if not n.lower().endswith(IMG_EXT) or '/images/' not in '/' + n:
                continue
            lab = re.sub('(^|/)images/', '\\1labels/', n, count=1).rsplit('.', 1)[0] + '.txt'
            les, teeth, other = ([], [], Counter())
            if lab in lab_set:
                for ln in tree.read(lab).decode().splitlines():
                    v = ln.split()
                    if len(v) < 5:
                        continue
                    c = cls.get(int(float(v[0])), '')
                    co = np.array(v[1:], dtype=float)
                    if len(co) == 4:
                        cx, cy, w, h = co
                        co = np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy - h / 2, cx + w / 2, cy + h / 2, cx - w / 2, cy + h / 2])
                    p = co[:len(co) // 2 * 2].reshape(-1, 2)
                    if any((c == k or c.startswith(k) for k in car)):
                        les.append((p, c))
                    elif c == tooth or c.startswith(tooth):
                        teeth.append(p)
                    else:
                        other[c] += 1
            stem = Path(n).stem
            parts = n.split('/')
            split = parts[parts.index('images') - 1] if 'images' in parts and parts.index('images') > 0 else ''
            tasks.append(dict(raw=n, ds='dai', stem=stem, base=_rf_base(stem), lesions=les, teeth=teeth, other=dict(other), official_split=split))
    seen = set()
    for t in tasks:
        t['dup_name'] = t['base'] in seen
        seen.add(t['base'])
        t['group'] = 'dai_' + _sanitize(t['base'])
    return tasks

def _coco_polys(a, w, h):
    seg = a.get('segmentation')
    out = []
    if isinstance(seg, list) and seg:
        for s in seg:
            if len(s) >= 6:
                out.append(np.array(s, dtype=float).reshape(-1, 2) / np.array([w, h]))
    if not out and a.get('bbox'):
        x, y, bw, bh = a['bbox']
        out.append(np.array([[x, y], [x + bw, y], [x + bw, y + bh], [x, y + bh]], dtype=float) / np.array([w, h]))
    return out

def _mask_instances(tree: RawTree, task: dict):
    import cv2
    m = np.array(Image.open(io.BytesIO(tree.read(task['mask']))))
    if m.ndim == 3:
        m = m[..., 0]
    H, W = m.shape
    les = []
    lesion_px = np.isin(m, task['car_v'])
    for v in task['car_v']:
        n, lab, stats, _ = cv2.connectedComponentsWithStats((m == v).astype(np.uint8), connectivity=8)
        for k in range(1, n):
            x, y, w, h, area = stats[k]
            if area < 4:
                continue
            les.append((np.array([[x, y], [x + w, y], [x + w, y + h], [x, y + h]], float) / [W, H], 'caries'))
    teeth_px = (np.isin(m, task['tooth_v']) | lesion_px).astype(np.uint8)
    er = cv2.erode(teeth_px, np.ones((5, 5), np.uint8), iterations=1)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(er, connectivity=4)
    teeth = []
    for k in range(1, n):
        x, y, w, h, area = stats[k]
        if area < 20:
            continue
        x, y, w, h = (x - 2, y - 2, w + 4, h + 4)
        teeth.append(np.array([[x, y], [x + w, y], [x + w, y + h], [x, y + h]], float) / [W, H])
    return (les, teeth)

def dai_labels(tree: RawTree, task: dict, W: int, H: int, tol: float, pcfg: dict):
    if task.get('fmt') == 'semantic_mask':
        task = dict(task)
        task['lesions'], task['teeth'] = _mask_instances(tree, task)
    lesion_polys = [p for p, _ in task['lesions']]
    boxes = np.array([poly_to_box(p) for p in lesion_polys]) if lesion_polys else np.zeros((0, 4))
    kept, n_oob, n_clip, keep = check_and_clip(boxes, tol)
    sub = [nm for (_, nm), k in zip(task['lesions'], keep) if k]
    tooth, n_unassigned = lesions_to_teeth(lesion_polys, task['teeth'], min_overlap=pcfg['tooth_min_overlap']) if lesion_polys else (np.zeros((0, 4)), 0)
    return dict(boxes=kept, sub=sub, n_oob=n_oob, n_clipped=n_clip, geom_note=task.get('fmt', ''), tooth=tooth, n_tooth_unassigned=n_unassigned, n_teeth=len(task['teeth']), other=task['other'])

def _process(tree: RawTree, task: dict, out_dir: Path, pcfg: dict):
    img = Image.open(io.BytesIO(tree.read(task['raw'])))
    img = ImageOps.exif_transpose(img).convert('RGB')
    W, H = img.size
    tol = pcfg['oob_tolerance']
    if task['ds'] == 'pk':
        lab = pk_labels(tree, task, W, H, tol)
    elif task['ds'] == 'ad':
        lab = ad_labels(tree, task, W, H, tol)
    else:
        lab = dai_labels(tree, task, W, H, tol, pcfg)
    s = min(1.0, pcfg['max_side'] / max(W, H))
    if s < 1.0:
        img = img.resize((max(1, round(W * s)), max(1, round(H * s))), Image.LANCZOS)
    iid = task['image_id']
    ip = out_dir / 'images' / f'{iid}.jpg'
    img.save(ip, quality=95)
    (out_dir / 'labels' / f'{iid}.txt').write_text(to_yolo_lines(lab['boxes']), encoding='utf-8')
    if 'tooth' in lab:
        (out_dir / 'labels_tooth' / f'{iid}.txt').write_text(to_yolo_lines(lab['tooth']), encoding='utf-8')
    return dict(image_id=iid, dataset=task['ds'], image=str(ip), width=img.size[0], height=img.size[1], orig_w=W, orig_h=H, group=task['group'], condition=task.get('condition', ''), view=task.get('view', ''), n_boxes=len(lab['boxes']), has_caries=int(len(lab['boxes']) > 0), n_oob=lab['n_oob'], n_clipped=lab['n_clipped'], geom_ok=int(lab['n_oob'] == 0), geom_note=lab.get('geom_note', ''), dup=int(task.get('dup_name', False)), sex=task.get('sex', ''), age=task.get('age', np.nan), raw_name=task['raw'], official_split=task.get('official_split', ''), dhash=dhash(img), n_tooth=len(lab['tooth']) if 'tooth' in lab else np.nan, n_tooth_unassigned=lab.get('n_tooth_unassigned', np.nan), _gt=dict(boxes=np.round(lab['boxes'], 5).tolist(), sub=lab['sub'], tooth=np.round(lab['tooth'], 5).tolist() if 'tooth' in lab else None), _other=lab.get('other', {}))

def assign_group_splits(df: pd.DataFrame, ratios, seed: int, trials: int, group_col: str='group') -> dict:
    g = df.groupby(group_col).agg(n=('image_id', 'size'), pos=('has_caries', 'sum')).reset_index()
    total, prev = (g.n.sum(), g.pos.sum() / max(1, g.n.sum()))
    names = np.array(['train', 'val', 'test'])
    best, best_score = (None, np.inf)
    for t in range(max(1, trials)):
        rng = np.random.default_rng(seed + t)
        order = rng.permutation(len(g))
        cum = np.cumsum(g.n.values[order]) / total
        lab = np.where(cum <= ratios[0] + 1e-09, 0, np.where(cum <= ratios[0] + ratios[1] + 1e-09, 1, 2))
        if len(set(lab)) < 3 and len(g) >= 3:
            continue
        score = 0.0
        for k in range(3):
            m = lab == k
            n_k = g.n.values[order][m].sum()
            score += abs(n_k / total - ratios[k])
            if n_k:
                score += abs(g.pos.values[order][m].sum() / n_k - prev)
        if score < best_score:
            best_score, best = (score, dict(zip(g[group_col].values[order], names[lab])))
    if best is None:
        best = {k: 'train' for k in g[group_col]}
    return best

def _thumb(path: str) -> np.ndarray:
    return np.asarray(Image.open(path).convert('L').resize((64, 64), Image.BILINEAR), dtype=np.float32)

def _merge_near_duplicates(df: pd.DataFrame, max_dist: int, max_group: int=60, max_mae: float=2.0):
    if df.empty or max_dist < 0:
        return (df, 0, 0, [])
    idx = df.index.to_numpy()
    hashes = df['dhash'].astype(str).tolist()
    ar = (df['width'] / df['height']).to_numpy()
    pairs = hamming_pairs(hashes, max_dist)
    uf = UF(list(df['group'].unique()))
    size = df.groupby('group').size().to_dict()
    n_merge, n_skip, dups, records = (0, 0, set(), [])
    for i, j in pairs:
        if abs(ar[i] - ar[j]) > 0.05:
            continue
        ri, rj = (df.loc[idx[i]], df.loc[idx[j]])
        mae = np.nan
        if hashes[i] == hashes[j] and ri['width'] == rj['width'] and (ri['height'] == rj['height']):
            mae = float(np.abs(_thumb(ri['image']) - _thumb(rj['image'])).mean())
        is_dup = np.isfinite(mae) and mae <= max_mae and (idx[i] not in dups)
        if is_dup:
            dups.add(idx[j])
        gi, gj = (uf.find(ri['group']), uf.find(rj['group']))
        merged = False
        if gi != gj:
            if size[gi] + size[gj] > max_group and (not is_dup):
                n_skip += 1
            else:
                n_merge += 1
                uf.union(gi, gj)
                size[uf.find(gi)] = size[gi] + size[gj]
                merged = True
        records.append(dict(kept=ri['image_id'], other=rj['image_id'], kept_raw=ri['raw_name'], other_raw=rj['raw_name'], hamming_bits=bin(int(hashes[i], 16) ^ int(hashes[j], 16)).count('1'), thumb_mae=mae, action='excluded duplicate' if is_dup else 'grouped' if merged or gi == gj else 'skipped (group cap)'))
    df = df.copy()
    df['group'] = df['group'].map(uf.find)
    df.loc[list(dups), 'dup'] = 1
    if n_skip:
        LOG.warning('near-duplicate merging: %d merge(s) skipped to keep groups <= %d images', n_skip, max_group)
    return (df, n_merge, len(dups), records)
CONVERTERS = {'pk': pk_tasks, 'ad': ad_tasks, 'dai': dai_tasks}

def is_prepared(ds: str, cfg: dict, paths: Paths) -> bool:
    out = paths.data / ds
    marker = out / '.done.json'
    return marker.exists() and jload(marker).get('hash') == config_hash(cfg) and (out / 'records.csv').exists() and (paths.gt / f'{ds}.json.gz').exists()

def prepare_dataset(ds: str, src: Path, cfg: dict, paths: Paths) -> pd.DataFrame:
    pcfg = cfg['prepare']
    out = paths.data / ds
    marker = out / '.done.json'
    h = config_hash(cfg)
    if is_prepared(ds, cfg, paths):
        LOG.info('dataset %s already prepared (%s)', ds, out)
        return pd.read_csv(out / 'records.csv', dtype={'group': str})
    tree = RawTree(src)
    LOG.info('preparing %s from %s (%d files)', ds, src, len(tree.names))
    tasks = CONVERTERS[ds](tree, pcfg)
    used_ann = {t.get('ann') for t in tasks}
    n_orphan = sum((1 for n in tree.names if '/Labelme/' in n and n.endswith('.json') and (n not in used_ann))) if ds == 'pk' else 0
    if not tasks:
        raise RuntimeError(f'no images found for {ds} in {src}')
    used = Counter()
    for t in tasks:
        base = f"{ds}_{_sanitize(t['stem'])}"[:120]
        used[base] += 1
        t['image_id'] = base if used[base] == 1 else f'{base}_{used[base]}'
    for sub in ('images', 'labels') + (('labels_tooth',) if ds == 'dai' else ()):
        (out / sub).mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(max_workers=pcfg.get('threads', 8)) as ex:
        recs = list(ex.map(lambda t: _process(tree, t, out, pcfg), tasks))
    gt = {r['image_id']: r.pop('_gt') for r in recs}
    other = Counter()
    for r in recs:
        other.update(r.pop('_other'))
    df = pd.DataFrame(recs)
    df, n_merge, n_dup, dup_pairs = _merge_near_duplicates(df, pcfg['dedup_hamming'], pcfg.get('max_group_images', 60))
    pd.DataFrame(dup_pairs, columns=['kept', 'other', 'kept_raw', 'other_raw', 'hamming_bits', 'thumb_mae', 'action']).to_csv(paths.gt / f'{ds}_near_duplicates.csv', index=False)
    excl_geom = pcfg['exclude_bad_geometry']
    df['usable'] = ((df['dup'] == 0) & ((df['geom_ok'] == 1) | (not excl_geom))).astype(int)
    use = df[df.usable == 1]
    ratios = pcfg['split_ratios']
    gmap = assign_group_splits(use, ratios, pcfg['seed'], pcfg['split_search_trials'])
    df['split'] = df['group'].map(gmap).fillna('excluded')
    df.loc[df.usable == 0, 'split'] = 'excluded'
    if ds == 'pk':
        imap = assign_group_splits(use, ratios, pcfg['seed'], pcfg['split_search_trials'], group_col='image_id')
        df['split_img'] = df['image_id'].map(imap).fillna('excluded')
        df.loc[df.usable == 0, 'split_img'] = 'excluded'
    else:
        df['split_img'] = df['split']
    rng = np.random.default_rng(pcfg['seed'])
    tr = df.index[(df.split == 'train') & (df.usable == 1)].to_numpy()
    pool = rng.choice(tr, size=min(pcfg['pool_size'], len(tr)), replace=False) if len(tr) else []
    df['pool'] = 0
    df.loc[pool, 'pool'] = 1
    df.to_csv(out / 'records.csv', index=False)
    jdump(gt, paths.gt / f'{ds}.json.gz')
    summary = dict(dataset=ds, source=str(src), n_found=len(df), n_orphan_annotations=n_orphan, n_dup=int(df.dup.sum()), n_near_dup_merges=n_merge, n_exact_dup=n_dup, n_geom_bad=int((df.geom_ok == 0).sum()), n_usable=int(df.usable.sum()), n_groups=int(df.loc[df.usable == 1, 'group'].nunique()), largest_group=int(df.loc[df.usable == 1].groupby('group').size().max()), n_pos=int(df.loc[df.usable == 1, 'has_caries'].sum()), n_boxes=int(df.loc[df.usable == 1, 'n_boxes'].sum()), n_clipped_boxes=int(df.n_clipped.sum()), other_classes=dict(other), splits=df.loc[df.usable == 1, 'split'].value_counts().to_dict())
    jdump(summary, paths.gt / f'{ds}_summary.json')
    jdump(dict(hash=h, n=len(df)), marker)
    LOG.info('prepared %s: %s', ds, {k: v for k, v in summary.items() if k not in ('source', 'other_classes')})
    return df

def prepare_all(cfg: dict, paths: Paths, sources: dict) -> pd.DataFrame:
    frames = []
    for ds, src in sources.items():
        if src is None:
            continue
        frames.append(prepare_dataset(ds, Path(src), cfg, paths))
    man = pd.concat(frames, ignore_index=True)
    prev = paths.manifest
    if prev.exists():
        old = pd.read_csv(prev, usecols=['image_id', 'split'])
        merged = old.merge(man[['image_id', 'split']], on='image_id', how='inner', suffixes=('_old', '_new'))
        changed = int((merged.split_old != merged.split_new).sum())
        if changed:
            LOG.warning("%d images changed split versus the previous session's manifest — results from earlier sessions are not comparable with this one", changed)
    man.to_csv(prev, index=False)
    return man

def load_manifest(paths: Paths) -> pd.DataFrame:
    return pd.read_csv(paths.manifest, dtype={'group': str, 'condition': str, 'view': str})

In [ ]:
%%writefile /tmp/dentx_src/dentx/report.py
from __future__ import annotations
import json
import platform
import sys
import time
import numpy as np
import pandas as pd
from .config import Paths
from .infer import dump_path, eval_sets_for
from .train import is_done
from .utils import jload, write_csv_and_tex
DATASET_INFO = {'pk': dict(country='Pakistan (Mithi, Sindh)', device='Samsung Galaxy A23 smartphone', protocol='box around each carious tooth (d/D)', population='10-24 y, school screening'), 'ad': dict(country='Russia', device='Canon 6D II DSLR, 100 mm macro, intraoral mirror', protocol='polygon around each lesion, Black classes I-VI (by location)', population='clinic patients (age, sex in file names)'), 'dai': dict(country='not reported (Roboflow/Kaggle)', device='mixed / not reported', protocol='polygon around each lesion (caries, cavity) + tooth instances', population='not reported')}

def env_record(cfg: dict, paths: Paths) -> dict:
    rec = dict(time=time.strftime('%Y-%m-%d %H:%M:%S'), python=sys.version.split()[0], platform=platform.platform(), profile=cfg['profile'])
    try:
        import torch
        rec['torch'] = torch.__version__
        rec['cuda'] = torch.version.cuda
        rec['gpus'] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    except Exception:
        pass
    try:
        import ultralytics
        rec['ultralytics'] = ultralytics.__version__
    except Exception:
        pass
    (paths.results / 'env.json').write_text(json.dumps(rec, indent=2), encoding='utf-8')
    return rec

def dataset_table(paths: Paths) -> pd.DataFrame:
    man = pd.read_csv(paths.manifest, dtype={'group': str})
    rows = []
    for ds, g in man.groupby('dataset'):
        s = jload(paths.gt / f'{ds}_summary.json') if (paths.gt / f'{ds}_summary.json').exists() else {}
        u = g[g.usable == 1]
        gt = jload(paths.gt / f'{ds}.json.gz')
        areas = [(b[2] - b[0]) * (b[3] - b[1]) for i in u.image_id for b in gt[i]['boxes']]
        info = DATASET_INFO.get(ds, {})
        rows.append(dict(dataset=ds, **info, images_found=len(g), excluded_geometry=int((g.geom_ok == 0).sum()), excluded_duplicates=int(g.dup.sum()), images_used=len(u), patients_or_groups=u.group.nunique(), caries_images_pct=100 * u.has_caries.mean(), boxes=int(u.n_boxes.sum()), median_box_area_pct=100 * float(np.median(areas)) if areas else np.nan, train=int((u.split == 'train').sum()), val=int((u.split == 'val').sum()), test=int((u.split == 'test').sum()), near_dup_merges=s.get('n_near_dup_merges', 0), labels_without_image=s.get('n_orphan_annotations', 0), clipped_boxes=s.get('n_clipped_boxes', 0)))
    df = pd.DataFrame(rows)
    write_csv_and_tex(df, paths.tables / 'table1_datasets', float_fmt='{:.1f}', caption='Datasets used')
    return df

def run_table(paths: Paths, plan: dict) -> pd.DataFrame:
    rows = []
    for rid, p in plan.items():
        d = jload(paths.runs / rid / 'done.json') if is_done(paths, rid) else {}
        rows.append(dict(run=rid, kind=p['kind'], model=p['model'], cfg=p['cfg'], n_train=p['n_train'], epochs_max=p['epochs'], epochs_run=d.get('epochs_run'), minutes=(d.get('wall_s') or np.nan) / 60, done=int(bool(d))))
    df = pd.DataFrame(rows)
    write_csv_and_tex(df, paths.tables / 'runs', float_fmt='{:.1f}', caption='Training runs')
    return df

def status(cfg: dict, paths: Paths, plan: dict) -> dict:
    man = pd.read_csv(paths.manifest, dtype={'group': str})
    avail = sorted(man.dataset.unique())
    runs_missing = [r for r in plan if not is_done(paths, r)]
    dumps_missing = []
    for r, p in plan.items():
        if not is_done(paths, r):
            continue
        for s in eval_sets_for(p, cfg, avail):
            if not dump_path(paths, r, s['name']).exists():
                dumps_missing.append(f"{r}:{s['name']}")
    return dict(complete=not runs_missing and (not dumps_missing), runs_total=len(plan), runs_missing=runs_missing, dumps_missing=len(dumps_missing), dumps_missing_examples=dumps_missing[:10])

def _md_table(df: pd.DataFrame, max_rows: int=30) -> str:
    if df is None or df.empty:
        return '_(none)_\n'
    df = df.head(max_rows)
    cols = list(df.columns)
    fmt = lambda v: f'{v:.3f}' if isinstance(v, (float, np.floating)) and np.isfinite(v) else '' if isinstance(v, float) else str(v)
    lines = ['| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)]
    lines += ['| ' + ' | '.join((fmt(v) for v in row)) + ' |' for row in df.itertuples(index=False)]
    return '\n'.join(lines) + '\n'

def write_results_md(cfg: dict, paths: Paths, plan: dict, summaries: dict, errors: dict) -> str:
    st = status(cfg, paths, plan)
    T = paths.tables

    def tab(name):
        f = T / f'{name}.csv'
        return pd.read_csv(f) if f.exists() else None
    parts = [f"# dentx results — profile `{cfg['profile']}`\n", f"Generated {time.strftime('%Y-%m-%d %H:%M')}. " + ('**All runs and dumps complete.**' if st['complete'] else f"**PARTIAL:** {len(st['runs_missing'])} of {st['runs_total']} runs and {st['dumps_missing']} inference dumps still missing — re-run the notebook with this output attached as an input to continue."), '\n']
    if st['runs_missing']:
        parts.append('Missing runs: ' + ', '.join(st['runs_missing'][:40]) + '\n')
    if errors:
        parts.append('## Analysis errors\n' + '\n'.join((f'- {k}: {v}' for k, v in errors.items())) + '\n')
    parts += ['## Datasets (Table 1)\n', _md_table(tab('table1_datasets')), '## E0 Replication and leakage\n', _md_table(tab('e0_replication_leakage')), _md_table(tab('e0_leakage_effect')), _md_table(tab('e0_cross_condition')), _md_table(tab('e0_calibration')), _md_table(tab('e0_eigencam_eibr')), '## E1 In-domain\n', _md_table(tab('e1_indomain')), _md_table(tab('e1_model_comparison')), '## E2 External validation (leave-one-dataset-out)\n', _md_table(tab('e2_generalisation_gap'), 60), _md_table(tab('e2_friedman')), _md_table(tab('e2_dai_tooth_level')), '## E3 Clinic or annotator? (protocol decomposition)\n', _md_table(tab('e3_protocol_summary')), '## E4 Label-free failure prediction\n', _md_table(tab('e4_transfer_mae')), _md_table(tab('e4_failure_detection')), _md_table(tab('e4_robustness_spearman')), '## E5 Few-label site onboarding\n', _md_table(tab('e5_fewshot_summary'), 60), '## Files\n- `tables/` CSV + LaTeX for every table; `all_metrics.csv` is the master table (one row per run x set x subset x ground-truth variant x threshold mode).\n- `figures/` PNG (300 dpi) + PDF.\n- `bootstrap_reps.pkl` patient-clustered bootstrap replicates for new paired tests.\n- `../dumps/` raw predictions; `../gt/` compact ground truth; metrics can be recomputed offline without images.\n']
    txt = '\n'.join(parts)
    (paths.results / 'RESULTS.md').write_text(txt, encoding='utf-8')
    (paths.results / 'status.json').write_text(json.dumps(st, indent=2), encoding='utf-8')
    return txt

In [ ]:
%%writefile /tmp/dentx_src/dentx/stats.py
from __future__ import annotations
import numpy as np
import pandas as pd
from scipy import stats as st

def paired_diff(a: np.ndarray, b: np.ndarray) -> dict:
    d = np.asarray(a, float) - np.asarray(b, float)
    d = d[np.isfinite(d)]
    if len(d) == 0:
        return dict(diff=np.nan, lo=np.nan, hi=np.nan, p=np.nan)
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return dict(diff=float(np.mean(d)), lo=float(np.quantile(d, 0.025)), hi=float(np.quantile(d, 0.975)), p=float(min(1.0, max(p, 1.0 / len(d)))))

def independent_diff(a: np.ndarray, b: np.ndarray, seed: int=0) -> dict:
    a, b = (np.asarray(a, float), np.asarray(b, float))
    n = min(len(a), len(b))
    rng = np.random.default_rng(seed)
    d = rng.permutation(a)[:n] - rng.permutation(b)[:n]
    return paired_diff(d, np.zeros_like(d))

def holm(pvals) -> np.ndarray:
    p = np.asarray(pvals, float)
    out = np.full_like(p, np.nan)
    ok = np.isfinite(p)
    idx = np.argsort(p[ok])
    m = ok.sum()
    adj = np.empty(m)
    run = 0.0
    for r, i in enumerate(idx):
        run = max(run, (m - r) * p[ok][i])
        adj[i] = min(1.0, run)
    out[ok] = adj
    return out

def friedman_nemenyi(df: pd.DataFrame, block_cols, model_col: str, value_col: str, higher_better: bool=True):
    piv = df.pivot_table(index=block_cols, columns=model_col, values=value_col, aggfunc='mean').dropna()
    k, n = (piv.shape[1], piv.shape[0])
    if k < 3 or n < 2:
        return (dict(n_blocks=n, k=k, chi2=np.nan, p=np.nan, cd=np.nan, ranks={}), piv)
    ranks = piv.rank(axis=1, ascending=not higher_better)
    chi2, p = st.friedmanchisquare(*[piv[c].values for c in piv.columns])
    q = st.studentized_range.ppf(0.95, k, np.inf) / np.sqrt(2)
    cd = q * np.sqrt(k * (k + 1) / (6.0 * n))
    return (dict(n_blocks=int(n), k=int(k), chi2=float(chi2), p=float(p), cd=float(cd), ranks=ranks.mean().sort_values().to_dict()), piv)

def spearman_ci(x, y, B: int=2000, seed: int=0) -> dict:
    x, y = (np.asarray(x, float), np.asarray(y, float))
    m = np.isfinite(x) & np.isfinite(y)
    x, y = (x[m], y[m])
    if len(x) < 4 or np.std(x) == 0 or np.std(y) == 0:
        return dict(rho=np.nan, lo=np.nan, hi=np.nan, p=np.nan, n=len(x))
    rho, p = st.spearmanr(x, y)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(B, len(x)))
    rx = st.rankdata(x[idx], axis=1)
    ry = st.rankdata(y[idx], axis=1)
    rx -= rx.mean(1, keepdims=True)
    ry -= ry.mean(1, keepdims=True)
    den = np.sqrt((rx ** 2).sum(1) * (ry ** 2).sum(1))
    with np.errstate(invalid='ignore', divide='ignore'):
        bs = (rx * ry).sum(1) / den
    bs = bs[np.isfinite(bs)]
    lo, hi = (np.quantile(bs, 0.025), np.quantile(bs, 0.975)) if len(bs) else (np.nan, np.nan)
    return dict(rho=float(rho), lo=float(lo), hi=float(hi), p=float(p), n=int(len(x)))

In [ ]:
%%writefile /tmp/dentx_src/dentx/synthetic.py
from __future__ import annotations
import io
import json
import zipfile
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw
VIEWS = ['Frontal', 'Left_Lateral', 'Right_Lateral', 'Maxillary_Occlusal', 'Mandibular']
PILOT_VIEWS = ['frontal', 'rightLateral', 'leftLateral', 'maxillary', 'mandibular']

def _ellipse(cx, cy, rx, ry, n=14):
    t = np.linspace(0, 2 * np.pi, n, endpoint=False)
    return np.stack([cx + rx * np.cos(t), cy + ry * np.sin(t)], 1)

def mouth(rng, W=320, H=240, p_caries=0.45):
    yy, xx = np.mgrid[0:H, 0:W]
    ang = rng.uniform(0, 2 * np.pi)
    grad = (np.cos(ang) * xx / W + np.sin(ang) * yy / H)[..., None] * rng.uniform(20, 70)
    base = np.array([190, 80, 95]) + rng.integers(-35, 35, 3)
    bg = np.clip(base + grad + rng.normal(0, 8, (H, W, 3)), 0, 255).astype(np.uint8)
    img = Image.fromarray(bg)
    d = ImageDraw.Draw(img)
    for _ in range(int(rng.integers(2, 6))):
        cx, cy, r = (rng.uniform(0, W), rng.uniform(0, H), rng.uniform(10, 50))
        v = int(rng.integers(40, 220))
        d.ellipse([cx - r, cy - r * 0.6, cx + r, cy + r * 0.6], fill=(v, int(v * 0.5), int(v * 0.55)))
    n = int(rng.integers(4, 9))
    span = W * rng.uniform(0.55, 0.9)
    x0 = rng.uniform(0, W - span)
    tw = span / n
    teeth, tpolys, lesions, owner = ([], [], [], [])
    yc, hh = (H * rng.uniform(0.35, 0.65), H * rng.uniform(0.2, 0.45))
    for i in range(n):
        x1 = x0 + i * tw + 2
        x2 = x1 + tw - 4
        y1 = yc - hh / 2 + rng.normal(0, 6)
        y2 = yc + hh / 2 + rng.normal(0, 6)
        shade = int(rng.integers(200, 250))
        d.rounded_rectangle([x1, y1, x2, y2], radius=int(rng.integers(2, 9)), fill=(shade, shade, shade - 12))
        teeth.append([x1, y1, x2, y2])
        tpolys.append(np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]]))
        if rng.random() < p_caries / 2:
            cx, cy = (rng.uniform(x1 + 4, x2 - 4), rng.uniform(y1 + 6, y2 - 6))
            poly = _ellipse(cx, cy, rng.uniform(3, max(3.5, tw / 4)), rng.uniform(3, 10))
            d.polygon([tuple(p) for p in poly], fill=(90, 55, 30))
            lesions.append(poly)
            owner.append(i)
    return (img, teeth, tpolys, lesions, owner)

def _jpg(img) -> bytes:
    b = io.BytesIO()
    img.save(b, 'JPEG', quality=90)
    return b.getvalue()

def make_pk(root: Path, rng, n_patients=24):
    base = root / 'pk_synth'
    for p in range(n_patients):
        t0 = 1730000000000 + p * 3600000
        for cond_dir, suf, dt in (('retractors', '01', 0), ('no_retractors', '00', 300000)):
            seq = 300 + 2 * p + (0 if suf == '01' else 1)
            for v, view in enumerate(VIEWS):
                stem = f'anonymous_003-007-{seq}-{suf}_{t0 + dt}_{view}_View'
                sub = Path('Dataset/Images') / cond_dir / view / (view if cond_dir == 'no_retractors' else '')
                forced = p == 0 and v == 0 and (cond_dir == 'retractors')
                img, teeth, _, les, owner = mouth(rng, p_caries=2.0 if forced else 0.45)
                (base / sub).mkdir(parents=True, exist_ok=True)
                (base / sub / f'{stem}.jpg').write_bytes(_jpg(img))
                if les:
                    shapes = []
                    for i in sorted(set(owner)):
                        x1, y1, x2, y2 = teeth[i]
                        if forced:
                            y2 = img.size[1] * 1.3
                        shapes.append(dict(label='D' if i % 3 else 'd', points=[[x1, y1], [x2, y2]], shape_type='rectangle'))
                    ann = base / 'Dataset/Annotations/Labelme' / cond_dir / view
                    ann.mkdir(parents=True, exist_ok=True)
                    (ann / f'{stem}.json').write_text(json.dumps(dict(shapes=shapes, imageWidth=img.size[0], imageHeight=img.size[1], imageData=None)))
    for s in range(2):
        t = 1726650000000 + s * 3000000
        for v, (view, dirv) in enumerate(zip(PILOT_VIEWS, VIEWS)):
            img, *_ = mouth(rng, p_caries=0.0)
            d = base / 'Dataset/Images/pilot' / dirv
            d.mkdir(parents=True, exist_ok=True)
            (d / f'anonymous-{view}View-{t + v * 120000}.jpg').write_bytes(_jpg(img))
    return base

def make_ad(root: Path, rng, n_patients=14):
    zp = root / 'AlphaDent.zip'
    names = {0: 'Abrasion', 1: 'Filling', 2: 'Crown', 3: 'Caries 1 class', 4: 'Caries 2 class', 5: 'Caries 3 class', 6: 'Caries 4 class', 7: 'Caries 5 class', 8: 'Caries 6 class'}
    with zipfile.ZipFile(zp, 'w') as z:
        z.writestr('yolo_seg_train.yaml', 'path: ./AlphaDent/\ntrain: images/train\nval: images/valid\nnames:\n' + ''.join((f'  {k}: {v}\n' for k, v in names.items())))
        for p in range(n_patients):
            split = 'valid' if p < 2 else 'train'
            for k in range(5):
                stem = f"p{p + 1:03d}_{('F' if p % 2 else 'M')}_{30 + p}_{k + 1:03d}"
                img, teeth, _, les, _ = mouth(rng, p_caries=0.9)
                W, H = img.size
                z.writestr(f'images/{split}/{stem}.jpg', _jpg(img))
                lines = []
                for poly in les:
                    c = int(rng.integers(3, 9))
                    lines.append(f'{c} ' + ' '.join((f'{x / W:.5f} {y / H:.5f}' for x, y in poly)))
                x1, y1, x2, y2 = teeth[0]
                lines.append('1 ' + ' '.join((f'{x / W:.5f} {y / H:.5f}' for x, y in [(x1, y1), (x2, y1), (x2, y2), (x1, y2)])))
                z.writestr(f'labels/{split}/{stem}.txt', '\n'.join(lines) + '\n')
        for k in range(3):
            img, *_ = mouth(rng)
            z.writestr(f'images/test/test_{k:03d}.jpg', _jpg(img))
    return zp

def make_dai(root: Path, rng, n=66, fmt: str='sly'):
    import tarfile
    splits = {'train': [], 'valid': [], 'test': []}
    for i in range(n):
        splits['train' if i < 46 else 'valid' if i < 56 else 'test'].append(i)
    splits['test'].append(3)
    cats = {1: 'Caries', 2: 'Cavity', 3: 'Crack', 4: 'Tooth'}
    items = {sp: [] for sp in splits}
    for sp, idx in splits.items():
        for i in idx:
            r = np.random.default_rng(1000 + i)
            img, teeth, tpolys, les, owner = mouth(r, p_caries=0.6)
            fn = f'img{i:03d}_jpg.rf.{rng.integers(1000000000.0):x}.jpg'
            objs = [(4, tp) for tp in tpolys] + [(1 if k % 2 == 0 else 2, lp) for k, lp in enumerate(les)]
            items[sp].append((fn, img, objs))
    if fmt == 'sly':
        stage = root / '_dai_sly_stage'
        stage.mkdir(parents=True, exist_ok=True)
        (stage / 'meta.json').write_text(json.dumps(dict(classes=[dict(title=v, shape='polygon') for v in cats.values()])))
        for sp, lst in items.items():
            (stage / sp / 'img').mkdir(parents=True, exist_ok=True)
            (stage / sp / 'ann').mkdir(parents=True, exist_ok=True)
            for fn, img, objs in lst:
                img.save(stage / sp / 'img' / fn, quality=90)
                ann = dict(size=dict(width=img.size[0], height=img.size[1]), objects=[dict(classTitle=cats[c], geometryType='polygon', points=dict(exterior=np.round(p).astype(int).tolist(), interior=[])) for c, p in objs])
                (stage / sp / 'ann' / f'{fn}.json').write_text(json.dumps(ann))
        tp = root / 'dentalai-DatasetNinja.tar'
        with tarfile.open(tp, 'w') as t:
            for f in sorted(stage.rglob('*')):
                if f.is_file():
                    t.add(f, arcname=f.relative_to(stage).as_posix())
        import shutil
        shutil.rmtree(stage)
        return tp
    if fmt == 'coco':
        base = root / 'dai_synth' / 'dentalai.v2i.coco-segmentation'
        for sp, lst in items.items():
            d = base / sp
            d.mkdir(parents=True, exist_ok=True)
            images, anns = ([], [])
            for j, (fn, img, objs) in enumerate(lst):
                img.save(d / fn, quality=90)
                images.append(dict(id=j, file_name=fn, width=img.size[0], height=img.size[1]))
                for c, p in objs:
                    anns.append(dict(id=len(anns), image_id=j, category_id=c, segmentation=[p.flatten().tolist()]))
            cat_list = [dict(id=0, name='dental-ai')] + [dict(id=k, name=v) for k, v in cats.items()]
            (d / '_annotations.coco.json').write_text(json.dumps(dict(images=images, annotations=anns, categories=cat_list)))
        return base
    base = root / 'dai_masks' / 'dentalai-2'
    for sp, lst in items.items():
        d = base / sp
        d.mkdir(parents=True, exist_ok=True)
        (d / '_classes.csv').write_text('Pixel Value, Class\n0, background\n1, Caries\n2, Cavity\n3, Crack\n4, Tooth\n')
        for fn, img, objs in lst:
            img.save(d / fn, quality=90)
            m = Image.new('L', img.size, 0)
            dr = ImageDraw.Draw(m)
            for c, p in sorted(objs, key=lambda o: o[0] != 4):
                dr.polygon([tuple(x) for x in p], fill=c)
            m.save(d / (fn[:-4] + '_mask.png'))
    return base

def make_all(root: Path, seed: int=0, dai_fmt: str='sly') -> dict:
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    return dict(pk=make_pk(root, rng), ad=make_ad(root, rng), dai=make_dai(root, rng, fmt=dai_fmt))

In [ ]:
%%writefile /tmp/dentx_src/dentx/train.py
from __future__ import annotations
import argparse
import hashlib
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
from .config import Paths, load_resolved
from .utils import LOG, jdump, jload, link_or_copy, setup_logging, stable_seed

def is_done(paths: Paths, run_id: str) -> bool:
    return (paths.runs / run_id / 'done.json').exists() and (paths.runs / run_id / 'best.pt').exists()

def _select(man: pd.DataFrame, datasets, flt=None) -> pd.DataFrame:
    m = man[(man.usable == 1) & man.dataset.isin(datasets)]
    for k, v in (flt or {}).items():
        m = m[m[k].astype(str) == str(v)]
    return m

def build_plan(cfg: dict, man: pd.DataFrame) -> dict:
    T = cfg['train']
    avail = set(man.dataset.unique())
    plan = {}
    for c in T['configs']:
        missing = [d for d in c['datasets'] if d not in avail]
        if missing:
            LOG.warning('skipping train config %s: dataset(s) %s unavailable', c['name'], missing)
            continue
        sel = _select(man, c['datasets'], c.get('filter'))
        split_col = c.get('split_col', 'split')
        n_train = int((sel[split_col] == 'train').sum())
        for m in c.get('models') or cfg['models']:
            ov = cfg.get('model_overrides', {}).get(m, {})
            rid = f"{m}__{c['name']}"
            plan[rid] = dict(run_id=rid, kind='main', model=m, cfg=c['name'], datasets=c['datasets'], filter=c.get('filter'), split_col=split_col, epochs=c.get('epochs', T['epochs']), patience=c.get('patience', T['patience']), batch=ov.get('batch', T['batch']), train_extra={k: v for k, v in ov.items() if k != 'batch'}, init=f'{m}.pt', val=True, priority=c.get('priority', 5), deps=[], n_train=n_train)
    fs = cfg['fewshot']
    for m in fs['models']:
        src = f"{m}__{fs['source']}"
        if src not in plan:
            continue
        for tgt in fs['targets']:
            if tgt not in avail:
                continue
            for k in fs['ks']:
                for s in fs['seeds']:
                    rid = f"fs__{m}__{fs['source']}2{tgt}__k{k}__s{s}"
                    plan[rid] = dict(run_id=rid, kind='fewshot', model=m, cfg=f'fs_{tgt}_k{k}_s{s}', datasets=[tgt], target=tgt, k=k, seed=s, source_run=src, epochs=fs['ft_epochs'], patience=fs['ft_epochs'] + 1, batch=min(plan[src]['batch'], k), init=f'@runs/{src}/best.pt', val=False, priority=9, deps=[src], n_train=k)
    pri = cfg['budget']['sec_per_img_epoch']
    oh = cfg['budget'].get('run_overhead_s', 120)
    for p in plan.values():
        p['est_s'] = oh + p['n_train'] * p['epochs'] * pri.get(p['model'], pri['default'])
    return plan

def drop_stale(plan: dict, paths: Paths) -> list:
    removed = []
    for d in sorted(paths.runs.glob('*')) if paths.runs.exists() else []:
        rid = d.name
        spec = plan.get(rid)
        done = d / 'done.json'
        stale = spec is None
        if not stale and done.exists():
            info = jload(done)
            stale = info.get('epochs_max') != spec['epochs'] or info.get('n_train') != spec['n_train'] or info.get('cfg') != spec['cfg'] or ((info.get('train_extra') or {}) != (spec.get('train_extra') or {}))
        if stale:
            shutil.rmtree(d, ignore_errors=True)
            removed.append(rid)
    for d in sorted(paths.dumps.glob('*')) if paths.dumps.exists() else []:
        if d.name in removed or not is_done(paths, d.name):
            shutil.rmtree(d, ignore_errors=True)
            if d.name not in removed:
                removed.append(d.name)
    if removed:
        LOG.warning('discarded %d stale run(s)/dump set(s) that do not match this plan: %s', len(removed), removed[:8])
    return removed

def fewshot_ids(man: pd.DataFrame, target: str, k: int, seed: int) -> list:
    pool = sorted(man[(man.dataset == target) & (man.pool == 1) & (man.usable == 1)].image_id)
    rng = np.random.default_rng(stable_seed('fewshot', target, seed))
    return [pool[i] for i in rng.permutation(len(pool))[:k]]

def build_cfg_dirs(plan: dict, man: pd.DataFrame, paths: Paths) -> None:
    by_id = man.set_index('image_id')
    done_cfgs = set()
    for p in plan.values():
        if p['cfg'] in done_cfgs:
            continue
        done_cfgs.add(p['cfg'])
        if p['kind'] == 'main':
            sel = _select(man, p['datasets'], p.get('filter'))
            parts = {'train': sorted(sel[sel[p['split_col']] == 'train'].image_id), 'val': sorted(sel[sel[p['split_col']] == 'val'].image_id)}
        else:
            ids = fewshot_ids(man, p['target'], p['k'], p['seed'])
            parts = {'train': ids, 'val': ids}
        d = paths.data / 'cfg' / p['cfg']
        sig = hashlib.md5(repr(parts).encode()).hexdigest()
        mk = d / '.sig'
        if mk.exists() and mk.read_text() == sig:
            continue
        if d.exists():
            shutil.rmtree(d)
        for split, ids in parts.items():
            for iid in ids:
                r = by_id.loc[iid]
                img = Path(r['image'])
                link_or_copy(img, d / 'images' / split / img.name)
                lab = img.parent.parent / 'labels' / (img.stem + '.txt')
                (d / 'labels' / split).mkdir(parents=True, exist_ok=True)
                shutil.copy2(lab, d / 'labels' / split / lab.name)
        (d / 'data.yaml').write_text(f'path: {d.as_posix()}\ntrain: images/train\nval: images/val\nnames:\n  0: caries\n', encoding='utf-8')
        mk.write_text(sig)
        LOG.info('config dir %s: %s', p['cfg'], {k: len(v) for k, v in parts.items()})

def disable_integrations() -> None:
    os.environ.setdefault('WANDB_MODE', 'disabled')
    os.environ.setdefault('WANDB_DISABLED', 'true')
    os.environ.setdefault('COMET_MODE', 'disabled')
    try:
        from ultralytics import settings
        settings.update({k: False for k in ('wandb', 'mlflow', 'comet', 'clearml', 'dvc', 'neptune', 'raytune', 'tensorboard', 'hub') if k in settings})
    except Exception:
        pass

def load_model(model_name: str, weights: str):
    from ultralytics import RTDETR, YOLO
    return RTDETR(weights) if 'rtdetr' in model_name else YOLO(weights)

def train_one(work: str, run_id: str, device: str) -> None:
    cfg = load_resolved(work)
    paths = Paths(cfg)
    setup_logging(paths.logs, f'train_{run_id}')
    spec = jload(paths.plan)[run_id]
    disable_integrations()
    paths.pretrained.mkdir(parents=True, exist_ok=True)
    os.chdir(paths.pretrained)
    init = spec['init']
    if init.startswith('@'):
        init = str(paths.work / init[1:])
    model = load_model(spec['model'], init)
    T = cfg['train']
    kw = dict(data=str(paths.data / 'cfg' / spec['cfg'] / 'data.yaml'), epochs=spec['epochs'], patience=spec['patience'], imgsz=T['imgsz'], batch=spec['batch'], workers=T['workers'], seed=T['seed'], deterministic=True, device=device, project=str(paths.train_tmp), name=run_id, exist_ok=True, val=spec['val'], plots=True, verbose=False)
    kw.update(T.get('extra') or {})
    kw.update(spec.get('train_extra') or {})
    t0 = time.time()
    model.train(**kw)
    wall = time.time() - t0
    save_dir = Path(model.trainer.save_dir)
    wdir = save_dir / 'weights'
    best = wdir / 'best.pt' if spec['val'] and (wdir / 'best.pt').exists() else wdir / 'last.pt'
    out = paths.runs / run_id
    out.mkdir(parents=True, exist_ok=True)
    for f in ('results.csv', 'args.yaml', 'results.png'):
        if (save_dir / f).exists():
            shutil.copy2(save_dir / f, out / f)
    epochs_run = len(pd.read_csv(save_dir / 'results.csv')) if (save_dir / 'results.csv').exists() else spec['epochs']
    import ultralytics
    shutil.copy2(best, out / 'best.pt.tmp')
    os.replace(out / 'best.pt.tmp', out / 'best.pt')
    jdump(dict(run_id=run_id, model=spec['model'], cfg=spec['cfg'], kind=spec['kind'], wall_s=wall, epochs_run=epochs_run, epochs_max=spec['epochs'], n_train=spec['n_train'], weights_from=best.name, train_extra=spec.get('train_extra') or {}, ultralytics=ultralytics.__version__, device=device), out / 'done.json')
    LOG.info('finished %s in %.1f min (%d epochs)', run_id, wall / 60, epochs_run)

def gpu_slots(cfg: dict):
    g = cfg['budget'].get('gpus', 'auto')
    if g == 'auto':
        try:
            import torch
            g = torch.cuda.device_count()
        except Exception:
            g = 0
    return [str(i) for i in range(int(g))] or ['cpu']

def run_queue(cfg: dict, paths: Paths, plan: dict, deadline: float, hard_deadline: float, module: str, items: list, label: str) -> None:
    slots = gpu_slots(cfg)
    running = {}
    pending = list(items)
    done_fn = (lambda i: is_done(paths, i)) if label == 'train' else lambda i: False
    finished = set()
    while pending or running:
        launched = False
        for slot in slots:
            if slot in running:
                continue
            now = time.time()
            for it in list(pending):
                iid, est, deps, args = it
                if done_fn(iid):
                    pending.remove(it)
                    continue
                if any((not is_done(paths, d) for d in deps)):
                    if any((d in [r[0] for r in running.values()] or d in [p[0] for p in pending] for d in deps)):
                        continue
                    pending.remove(it)
                    LOG.warning('%s %s dropped: dependency missing', label, iid)
                    continue
                if now + est * 1.15 > deadline:
                    continue
                env = dict(os.environ)
                dev = 'cpu'
                if slot != 'cpu':
                    env['CUDA_VISIBLE_DEVICES'] = slot
                    dev = '0'
                log = open(paths.logs / f"{label}_{iid.replace('/', '_')}.log", 'w', encoding='utf-8')
                cmd = [sys.executable, '-m', module, '--work', str(paths.work), '--device', dev] + args
                pkg_root = str(Path(__file__).resolve().parent.parent)
                env['PYTHONPATH'] = pkg_root + os.pathsep + env.get('PYTHONPATH', '')
                proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, cwd=pkg_root)
                running[slot] = (iid, proc, now, log)
                pending.remove(it)
                launched = True
                LOG.info('[%s] start %s on %s (est %.0f min)', label, iid, slot, est / 60)
                break
        time.sleep(5 if running else 0.1)
        ended = False
        for slot, (iid, proc, t0, log) in list(running.items()):
            rc = proc.poll()
            if rc is None and time.time() > hard_deadline:
                LOG.warning('[%s] %s killed at the session deadline', label, iid)
                proc.terminate()
                try:
                    proc.wait(60)
                except subprocess.TimeoutExpired:
                    proc.kill()
                rc = -9
            if rc is not None:
                log.close()
                del running[slot]
                ended = True
                LOG.info('[%s] %s ended rc=%s after %.1f min', label, iid, rc, (time.time() - t0) / 60)
                if rc == 0:
                    finished.add(iid)
                else:
                    LOG.warning('[%s] %s failed; see logs/%s_%s.log', label, iid, label, iid)
        if not running and (not launched) and (not ended) and pending:
            LOG.warning('[%s] %d item(s) left for the next session (time budget or failed dependencies)', label, len(pending))
            break

def run_training(cfg: dict, paths: Paths, plan: dict, deadline: float, hard_deadline: float) -> None:
    _refine_estimates(plan, paths, cfg)
    items = [(rid, p['est_s'], p['deps'], ['--run', rid]) for rid, p in sorted(plan.items(), key=lambda kv: (kv[1]['priority'], -kv[1]['est_s'])) if not is_done(paths, rid)]
    if not items:
        LOG.info('all %d training runs already done', len(plan))
        return
    LOG.info('training queue: %d of %d runs to do', len(items), len(plan))
    run_queue(cfg, paths, plan, deadline, hard_deadline, 'dentx.train', items, 'train')

def _refine_estimates(plan: dict, paths: Paths, cfg: dict) -> None:
    measured = {}
    for rid, p in plan.items():
        f = paths.runs / rid / 'done.json'
        if f.exists() and p['kind'] == 'main':
            d = jload(f)
            oh = cfg['budget'].get('run_overhead_s', 120)
            rate = max(0.0, d['wall_s'] - oh) / max(1, d['n_train'] * d['epochs_run'])
            measured[p['model']] = max(measured.get(p['model'], 0), rate)
    oh = cfg['budget'].get('run_overhead_s', 120)
    for p in plan.values():
        if p['model'] in measured:
            p['est_s'] = oh + p['n_train'] * p['epochs'] * measured[p['model']]

def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--work', required=True)
    ap.add_argument('--run', required=True)
    ap.add_argument('--device', default='0')
    a = ap.parse_args(argv)
    train_one(a.work, a.run, a.device)
if __name__ == '__main__':
    main()

In [ ]:
%%writefile /tmp/dentx_src/dentx/utils.py
from __future__ import annotations
import gzip
import hashlib
import json
import logging
import os
import shutil
import sys
import time
import zipfile
from pathlib import Path
import numpy as np
LOG = logging.getLogger('dentx')

def setup_logging(log_dir: Path | None=None, name: str='dentx') -> logging.Logger:
    LOG.setLevel(logging.INFO)
    fmt = logging.Formatter('%(asctime)s %(levelname)s %(message)s', '%H:%M:%S')
    if not any((isinstance(h, logging.StreamHandler) and (not isinstance(h, logging.FileHandler)) for h in LOG.handlers)):
        sh = logging.StreamHandler(sys.stdout)
        sh.setFormatter(fmt)
        LOG.addHandler(sh)
    if log_dir is not None:
        log_dir.mkdir(parents=True, exist_ok=True)
        target = str(log_dir / f'{name}.log')
        if not any((isinstance(h, logging.FileHandler) and h.baseFilename == os.path.abspath(target) for h in LOG.handlers)):
            fh = logging.FileHandler(target, encoding='utf-8')
            fh.setFormatter(fmt)
            LOG.addHandler(fh)
    LOG.propagate = False
    return LOG

def jdump(obj, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    data = json.dumps(obj, default=_json_default).encode('utf-8')
    if path.suffix == '.gz':
        with gzip.open(tmp, 'wb', compresslevel=6) as f:
            f.write(data)
    else:
        tmp.write_bytes(data)
    os.replace(tmp, path)

def jload(path: Path):
    path = Path(path)
    if path.suffix == '.gz':
        with gzip.open(path, 'rb') as f:
            return json.loads(f.read().decode('utf-8'))
    return json.loads(path.read_text(encoding='utf-8'))

def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    raise TypeError(type(o))

def stable_seed(*parts) -> int:
    h = hashlib.md5('|'.join((str(p) for p in parts)).encode()).hexdigest()
    return int(h[:8], 16) % (2 ** 31 - 1)

def link_or_copy(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        return
    try:
        os.symlink(os.path.abspath(src), dst)
        return
    except (OSError, NotImplementedError):
        pass
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

class Timer:

    def __init__(self):
        self.t0 = time.time()

    def elapsed_h(self) -> float:
        return (time.time() - self.t0) / 3600.0

class RawTree:

    def __init__(self, root: Path):
        import tarfile
        import threading
        self.root = Path(root)
        self.kind = 'dir'
        self._local = threading.local()
        if self.root.is_file() and zipfile.is_zipfile(self.root):
            self.kind = 'zip'
            with zipfile.ZipFile(self.root) as zf:
                self.names = [n for n in zf.namelist() if not n.endswith('/')]
        elif self.root.is_file() and tarfile.is_tarfile(self.root):
            self.kind = 'tar'
            with tarfile.open(self.root, 'r:') as tf:
                self._index = {m.name.lstrip('./'): (m.offset_data, m.size) for m in tf.getmembers() if m.isfile()}
            self.names = sorted(self._index)
        else:
            self.names = sorted((p.relative_to(self.root).as_posix() for p in self.root.rglob('*') if p.is_file()))

    @property
    def is_zip(self) -> bool:
        return self.kind == 'zip'

    def read(self, name: str) -> bytes:
        if self.kind == 'zip':
            zf = getattr(self._local, 'zf', None)
            if zf is None:
                zf = self._local.zf = zipfile.ZipFile(self.root)
            return zf.read(name)
        if self.kind == 'tar':
            fh = getattr(self._local, 'fh', None)
            if fh is None:
                fh = self._local.fh = open(self.root, 'rb')
            off, size = self._index[name]
            fh.seek(off)
            return fh.read(size)
        return (self.root / name).read_bytes()

    def __repr__(self):
        return f'RawTree({self.root}, kind={self.kind}, n={len(self.names)})'

def write_csv_and_tex(df, path_no_ext: Path, float_fmt: str='{:.3f}', caption: str='') -> None:
    path_no_ext = Path(path_no_ext)
    path_no_ext.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path_no_ext.with_suffix('.csv'), index=False)

    def cell(v):
        if isinstance(v, (float, np.floating)):
            return '' if np.isnan(v) else float_fmt.format(v)
        return str(v).replace('_', '\\_').replace('%', '\\%').replace('&', '\\&')
    cols = list(df.columns)
    lines = ['\\begin{table}[ht]', '\\centering']
    if caption:
        lines.append('\\caption{' + caption + '}')
    lines += ['\\begin{tabular}{' + 'l' * len(cols) + '}', '\\hline', ' & '.join((cell(c) for c in cols)) + ' \\\\', '\\hline']
    for _, row in df.iterrows():
        lines.append(' & '.join((cell(row[c]) for c in cols)) + ' \\\\')
    lines += ['\\hline', '\\end{tabular}', '\\end{table}']
    path_no_ext.with_suffix('.tex').write_text('\n'.join(lines), encoding='utf-8')

In [ ]:
%%writefile /tmp/dentx_src/dentx/config.yaml
profile: full
paths:
  work: auto
  data: auto
  raw: auto
  inputs: auto
datasets:
  pk:
    title: AKU intraoral caries (Pakistan, smartphone)
    urls:
    - https://zenodo.org/api/records/14827784/files/Dataset.zip/content
    archive: Dataset.zip
  ad:
    title: AlphaDent (Russia, DSLR + mirror)
    urls:
    - https://zenodo.org/api/records/16582489/files/AlphaDent.zip/content
    - https://huggingface.co/datasets/ZFTurbo/AlphaDent/resolve/main/AlphaDent.zip
    archive: AlphaDent.zip
  dai:
    title: DentalAI (Roboflow export; instance polygons via Dataset Ninja, CC BY 4.0)
    urls:
    - https://assets.supervisely.com/remote/eyJsaW5rIjogInMzOi8vc3VwZXJ2aXNlbHktZGF0YXNldHMvMjA4NF9EZW50YWxhaS9kZW50YWxhaS1EYXRhc2V0TmluamEudGFyIiwgInNpZyI6ICJybmQ4ZG5hUXRtTjBXMUw2TndsUHBFLzhFd3Z1NEFBZUtFV3VCQ0RMcTFzPSJ9?response-content-disposition=attachment%3B%20filename%3D%22dentalai-DatasetNinja.tar%22
    archive: dentalai-DatasetNinja.tar
    kaggle_slug: pawanvalluri/dental-segmentation
prepare:
  max_side: 1280
  split_ratios:
  - 0.7
  - 0.15
  - 0.15
  seed: 42
  split_search_trials: 300
  oob_tolerance: 0.02
  exclude_bad_geometry: true
  dedup_hamming: 10
  max_group_images: 60
  pk_link_window_s: 1800
  pk_link_seq_delta: 1
  pilot_gap_s: 900
  ad_caries_name: caries
  dai_caries_names:
  - caries
  - cavity
  dai_tooth_name: tooth
  tooth_min_overlap: 0.1
  pool_size: 200
  cleanup_raw: false
  threads: 8
models:
- yolov8s
- yolo11s
- yolo12s
- yolo26s
- rtdetr-l
model_overrides:
  rtdetr-l:
    batch: 8
    optimizer: AdamW
    lr0: 0.0001
    weight_decay: 0.0001
train:
  imgsz: 640
  epochs: 80
  patience: 15
  batch: 16
  workers: 4
  seed: 0
  extra: {}
  configs:
  - name: pk
    datasets:
    - pk
    priority: 1
  - name: ad
    datasets:
    - ad
    priority: 1
  - name: dai
    datasets:
    - dai
    priority: 1
  - name: pk-ad
    datasets:
    - pk
    - ad
    priority: 2
  - name: pk-dai
    datasets:
    - pk
    - dai
    priority: 2
  - name: ad-dai
    datasets:
    - ad
    - dai
    priority: 2
  - name: pkR
    datasets:
    - pk
    filter:
      condition: retractor
    models:
    - yolo11s
    - yolo12s
    priority: 3
  - name: pkN
    datasets:
    - pk
    filter:
      condition: no_retractor
    models:
    - yolo11s
    - yolo12s
    priority: 3
  - name: pkimg
    datasets:
    - pk
    split_col: split_img
    models:
    - yolo11s
    priority: 3
  - name: pkimg20
    datasets:
    - pk
    split_col: split_img
    models:
    - yolo11s
    - yolo12s
    epochs: 20
    patience: 100
    priority: 4
infer:
  conf: 0.01
  nms_iou: 0.7
  max_det: 50
  batch: 16
  features: true
  feature_layer: -2
  flip: true
  flip_corrupted: true
eval:
  thr_fixed: 0.25
  iou: 0.5
  iomin: 0.5
  ece_bins: 10
  bootstrap: 2000
  target_sensitivity: 0.9
  failure_drop: 0.1
corruptions:
  types:
  - gaussian_blur
  - brightness
  - contrast
  - gaussian_noise
  - jpeg
  severities:
  - 1
  - 3
  - 5
  sources:
  - pk
  - ad
  - dai
cam:
  models:
  - yolo11s
  - yolo12s
  layer: -2
  n_examples: 6
fewshot:
  models:
  - yolo11s
  source: pk
  targets:
  - ad
  - dai
  ks:
  - 10
  - 25
  - 50
  - 100
  seeds:
  - 0
  - 1
  - 2
  ft_epochs: 30
  alpha: 0.1
  alphas:
  - 0.1
  - 0.2
  - 0.3
budget:
  hours: 11.3
  infer_reserve_h: 3.0
  analysis_reserve_h: 1.0
  gpus: auto
  run_overhead_s: 120
  sec_per_img_epoch:
    yolov8s: 0.012
    yolo11s: 0.014
    yolo12s: 0.022
    yolo26s: 0.019
    rtdetr-l: 0.075
    default: 0.025
package:
  include_weights: true
profiles:
  free:
    models:
    - yolov8s
    - yolo11s
    - yolo12s
    - rtdetr-l
    train:
      epochs: 50
      patience: 10
      workers: 2
      configs:
      - name: pk
        datasets:
        - pk
        priority: 1
      - name: ad
        datasets:
        - ad
        priority: 1
      - name: dai
        datasets:
        - dai
        priority: 1
      - name: pk-ad
        datasets:
        - pk
        - ad
        models:
        - yolo11s
        priority: 2
      - name: pk-dai
        datasets:
        - pk
        - dai
        models:
        - yolo11s
        priority: 2
      - name: ad-dai
        datasets:
        - ad
        - dai
        models:
        - yolo11s
        priority: 2
      - name: pkR
        datasets:
        - pk
        filter:
          condition: retractor
        models:
        - yolo11s
        priority: 3
      - name: pkN
        datasets:
        - pk
        filter:
          condition: no_retractor
        models:
        - yolo11s
        priority: 3
      - name: pkimg20
        datasets:
        - pk
        split_col: split_img
        models:
        - yolo11s
        - yolo12s
        epochs: 20
        patience: 100
        priority: 3
      - name: pk20
        datasets:
        - pk
        models:
        - yolo11s
        - yolo12s
        epochs: 20
        patience: 100
        priority: 3
    infer:
      flip_corrupted: false
    cam:
      models:
      - yolo11s
    fewshot:
      ks:
      - 10
      - 25
      - 50
      - 100
      seeds:
      - 0
      - 1
      - 2
      ft_epochs: 20
    budget:
      hours: 11.3
      infer_reserve_h: 2.0
      analysis_reserve_h: 0.8
  check:
    models:
    - yolo11s
    train:
      epochs: 3
      patience: 100
      configs:
      - name: pk
        datasets:
        - pk
        priority: 1
      - name: ad
        datasets:
        - ad
        priority: 1
      - name: dai
        datasets:
        - dai
        priority: 1
    corruptions:
      types:
      - gaussian_blur
      - jpeg
      severities:
      - 3
    infer:
      flip_corrupted: false
    cam:
      models:
      - yolo11s
      n_examples: 3
    fewshot:
      ks:
      - 10
      seeds:
      - 0
      ft_epochs: 3
    eval:
      bootstrap: 500
    budget:
      hours: 2.0
      infer_reserve_h: 0.5
      analysis_reserve_h: 0.3
    package:
      include_weights: false
  smoke:
    models:
    - yolo11n
    - yolov8n
    - yolo12n
    model_overrides: {}
    prepare:
      max_side: 320
      split_search_trials: 20
      pool_size: 12
      threads: 4
    train:
      imgsz: 160
      epochs: 1
      patience: 100
      batch: 8
      workers: 0
      configs:
      - name: pk
        datasets:
        - pk
        priority: 1
      - name: ad
        datasets:
        - ad
        priority: 1
      - name: dai
        datasets:
        - dai
        priority: 1
      - name: pk-ad
        datasets:
        - pk
        - ad
        models:
        - yolo11n
        priority: 2
      - name: pkR
        datasets:
        - pk
        filter:
          condition: retractor
        models:
        - yolo11n
        priority: 3
      - name: pkN
        datasets:
        - pk
        filter:
          condition: no_retractor
        models:
        - yolo11n
        priority: 3
      - name: pkimg
        datasets:
        - pk
        split_col: split_img
        models:
        - yolo11n
        priority: 3
    infer:
      batch: 8
    corruptions:
      types:
      - gaussian_blur
      - jpeg
      severities:
      - 3
    cam:
      models:
      - yolo11n
      n_examples: 2
    fewshot:
      models:
      - yolo11n
      ks:
      - 4
      - 8
      seeds:
      - 0
      - 1
      ft_epochs: 1
    eval:
      bootstrap: 200
    budget:
      hours: 2.0
      infer_reserve_h: 0.3
      analysis_reserve_h: 0.2
    package:
      include_weights: false

In [ ]:
extra = ("--set " + EXTRA) if EXTRA.strip() else ""
!cd /tmp/dentx_src && python -m dentx.pipeline --profile $PROFILE --hours $HOURS $extra

In [ ]:
import glob, os
from IPython.display import Markdown, display
zips = sorted(glob.glob("/kaggle/working/dentx_results_*.zip"))
print("\n".join(f"{z}  {os.path.getsize(z)/2**20:.1f} MB" for z in zips))
p = "/kaggle/working/dentx_run/results/RESULTS.md"
if os.path.exists(p):
    display(Markdown(open(p, encoding="utf-8").read()))